# OPSD: an on-policy student with a privileged frozen teacher

**Model:** `Qwen/Qwen3.5-0.8B` · **Domains:** selectable math, coding, tool use · **Evaluation:** paired before/after per domain

On-Policy Self-Distillation (OPSD) samples answers from the ordinary student and scores those exact prefixes with a teacher that can see a verified solution. Here the teacher is a frozen copy of the initial Qwen checkpoint, while the student changes. Jensen–Shannon divergence transfers the teacher's informed next-token distribution to a student that must solve the question without the reference.

[OPSD](https://arxiv.org/abs/2601.18734) motivates on-policy distillation with privileged information. This repository implements the simple full-vocabulary JSD/reverse-KL variant; it does not implement all later paper variants, pointwise divergence clipping, or the full published evaluation setup.

**Standalone Colab edition.** Upload this file alone and run it.

**Recipe:** Sample ordinary student answer → initial frozen teacher with verified reference → JSD on sampled prefixes → update student → fresh rollout

You will inspect the raw and normalized dataset, verify the reward, work through the masked tensor shapes and exact objective, run real training, inspect its buffer, and evaluate seed and trained checkpoints. The small mathematical examples run on CPU; the full pretrained training stage needs an appropriate training environment. This standalone recipe embeds every required training and evaluation function. No repository import or checkout is used.

This standalone recipe supports selectable **math, Python coding, and single-turn tool-use** domains, including mixed-domain training, per-domain validation, and paired before/after final benchmarks. The loss and requested model are unchanged; dataset adapters and verifiers supply domain-specific learning signals.

## 1. Run this file alone in Google Colab

Upload this **one `.ipynb` file** through Colab's **File → Upload notebook**. Select **Runtime → Change runtime type → GPU**, then run cells in order. No Git clone, repository checkout, local module, trainer package, helper file, or external requirements file is needed. The next single cell lists all external libraries, writes `requirements-standalone.txt`, and installs them into the active notebook environment. Colab already provides the notebook interface; installing JupyterLab is unnecessary.

Run the installation cell before importing the training libraries. If it upgrades a package already imported in your session, restart the session and run from the first cell again. Keep the CUDA-compatible PyTorch build supplied by the runtime when it satisfies the requirement. Models and official datasets download directly from Hugging Face on first use. The default public data/models need no token; optional xLAM requires accepting its dataset terms and HF authentication.

The configuration starts with 16 training and 8 validation questions **per selected domain**, one training round, a 256-token response budget, and a 32-question test **subset per benchmark**. All three domains are selected by default. Set `BENCHMARK_LIMIT=None` for every prepared test row; source and context exclusions remain recorded. To match the longer original recipes, set `MAX_TOKENS=512` for Qwen or `256` for LFM. Record these budget changes when comparing results.

All functions are ordinary visible Python cells below. Generation, training, and evaluation load their own models inside functions, release them on return, and clear unused cached memory between stages. OPSD's frozen teacher defaults to CPU; only its current vocabulary chunks transfer to the GPU for the loss. This reduces GPU memory use but is slower. `REFERENCE_DEVICE = torch.device("cuda")` can speed it up when enough memory is available. GRPO's smaller frozen reference defaults to the training device. Trainable weights remain FP32, with CUDA autocast and gradient scaling when needed.

Colab GPU types, session duration, and available memory vary; full-parameter Qwen training can need a larger runtime even with checkpointing and small projection chunks. The notebook prints the actual device. Optional Qwen CUDA kernels are not required for correctness; the default PyTorch fallback avoids a compilation dependency but can be slower. See the [Colab runtime FAQ](https://research.google.com/colaboratory/faq.html) and [official Qwen architecture notes](https://huggingface.co/docs/transformers/model_doc/qwen3_5).

Artifacts live in `RUN_DIR`, normally `/content/rl_tutorials/<method>`. Colab's local disk is temporary. Change that path to mounted Drive if you want persistence, or use the final optional download cell. Rerunning with the same directory replaces that experiment. Committed notebook outputs are cleared; no test score is asserted in advance.

In [ ]:
# The complete external requirements live in this single cell; no repo file is needed.
import os
import subprocess
import sys
from importlib import metadata
from pathlib import Path

REQUIREMENTS = ["torch>=2.6", "transformers>=5.3,<6", "datasets>=3.3,<6",
                "matplotlib>=3.9,<4", "ipython>=7,<10", "tokenizers>=0.22,<1",
                "jsonschema>=4.23,<5", "huggingface-hub>=0.34,<2"]
requirements_path = Path("requirements-standalone.txt")
requirements_path.write_text("\n".join(REQUIREMENTS) + "\n")
loaded_versions = {name: getattr(sys.modules[name], "__version__", None)
                   for name in ("torch", "transformers", "datasets") if name in sys.modules}
# Only automated offline validation sets this variable. Normal Colab runs install here.
if os.environ.get("STANDALONE_SKIP_INSTALL") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)])
changed = [name for name, version in loaded_versions.items()
           if version and metadata.version(name) != version]
if changed:
    raise RuntimeError(f"Updated already-imported packages: {changed}. Restart the session and run from this cell again.")
print("Requirements:", *REQUIREMENTS, sep="\n")

In [ ]:
import ast
import gc
import hashlib
import json
import math
import random
import re
import tempfile
import time
from collections import Counter
from collections.abc import Iterable, Iterator, Sequence
from dataclasses import dataclass
from decimal import Decimal, InvalidOperation
from fractions import Fraction
from types import SimpleNamespace
from typing import Any

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from IPython.display import Markdown, display
from jsonschema import Draft7Validator, SchemaError, ValidationError
from torch import nn
from torch.utils.checkpoint import checkpoint
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoModelForImageTextToText,
    AutoTokenizer,
    GenerationConfig,
)


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

### Select domains, datasets, and mixture size

Edit only the configuration cell for a first run:

```python
DOMAINS = "all"                          # math + coding + tool_use
# DOMAINS = ["coding"]                   # Python only
# DOMAINS = ["tool_use"]                 # function calling only
# DOMAINS = ["math"]                     # the original arithmetic recipe
# DOMAINS = ["coding", "tool_use"]        # combine any supported domains
# TRAIN_SOURCES["math"] = ["gsm8k", "svamp"]
# TRAIN_SOURCES["tool_use"] = ["hermes", "xlam"]  # xLAM needs accepted HF terms/token
# TRAIN_LIMITS = {"math": 32, "coding": 16, "tool_use": 8}  # custom mixture
```

`TRAIN_LIMITS` and `VAL_LIMITS` are **counts per selected domain**, not global totals. The default is 16 training / 8 validation questions per domain, so `"all"` fits 48 unique tasks. Every domain has its own JSONL files; selected training rows are combined and deterministically shuffled into `train.jsonl`. Selecting several sources within one domain samples from their combined eligible pool, in proportion to available rows. The counts control the domain mix, not source balance. For $n_d$ selected tasks and $K$ responses per task, the rollout fraction is approximately $w_d=n_d/\sum_j n_j$ when every group has $K$ responses; different output lengths and batch grouping can still affect token-weighted gradients.

`MODEL_REVISION` and `DATA_REVISIONS` accept HF commit SHAs. `ROUNDS` creates fresh policy data each round. `SAMPLES=4` for GRPO and 1 otherwise; `MAX_TOOL_CALLS=2` keeps this starter's tool actions short. `MAX_TOKENS=256` applies equally before and after training; raise it for longer code and report truncation. `MAX_SEQ_LENGTH` covers student and private teacher contexts. Long training tasks are excluded with recorded counts; actual SDPO feedback is checked again after generation. `CHUNK_SIZE`, `ACCUMULATION`, and method-specific `HP` retain the original optimizer behavior. Output directories include the domain selection and a configuration digest to keep different experiments apart.

`BENCHMARK_LIMIT=32` evaluates a deterministic subset **per benchmark**. `None` evaluates every prepared benchmark row, including the recorded context filter. It does not restore filtered-out tasks. `TOOL_TEST_LIMIT=128` sets the separate reserved source-distribution tool test size. Tune only on validation, then freeze configuration before the final benchmark.

Hermes, MBPP, HumanEval, GSM8K, SVAMP, and BFCL are the default public sources. xLAM is optional: first accept its Hugging Face access terms, then set `HF_TOKEN` using a Colab secret or `huggingface_hub.login()`; never put a token in notebook outputs. For a Colab secret named `HF_TOKEN`, run the following only when selecting xLAM:

```python
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
```

A gated download fails clearly rather than silently changing your selected dataset. All downloads are data; no repository clone/import is needed.

`STANDALONE_SMOKE=1` creates tiny real model architectures and synthetic examples **in all selected domains**, performs two update rounds, and labels every score as a smoke check. `STANDALONE_SKIP_INSTALL=1` is only for an environment with the dependency cell already installed. Normal Colab runs leave both unset.

In [ ]:
ALGORITHM = "opsd"
MODEL_ID = "Qwen/Qwen3.5-0.8B"
MODEL_REVISION = "main"  # Use a model commit SHA to pin its weights/tokenizer.
DOMAINS = "all"  # Or ["math"], ["coding"], ["tool_use"], any combination.
TRAIN_SOURCES = {"math": ["gsm8k"], "coding": ["mbpp"], "tool_use": ["hermes"]}
SELECTED_DOMAINS = list(TRAIN_SOURCES) if DOMAINS == "all" else list(dict.fromkeys(DOMAINS))
if not SELECTED_DOMAINS or any(d not in TRAIN_SOURCES for d in SELECTED_DOMAINS):
    raise ValueError("DOMAINS must be 'all' or a nonempty list of supported domains")
DATA_REVISIONS = {name: "main" for name in ("gsm8k", "svamp", "mbpp", "humaneval", "hermes", "xlam", "bfcl_simple")}
SMOKE = os.environ.get("STANDALONE_SMOKE") == "1"
RUN_DIR = Path(os.environ.get("STANDALONE_RUN_ROOT", "/content/rl_tutorials" if Path("/content").exists() else "rl_tutorials")) / ALGORITHM
RUN_DIR = RUN_DIR.resolve()
RUN_DIR.mkdir(parents=True, exist_ok=True)
MODEL = MODEL_ID
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# OPSD's frozen Qwen teacher defaults to CPU to leave more room for actor/Adam states.
REFERENCE_DEVICE = torch.device("cpu" if ALGORITHM == "opsd" else DEVICE.type)
SEED = 7
TRAIN_LIMITS = {d: (2 if SMOKE else 16) for d in SELECTED_DOMAINS}
VAL_LIMITS = {d: (1 if SMOKE else 8) for d in SELECTED_DOMAINS}
TOOL_TEST_LIMIT, MAX_TOOL_CALLS = 128, 2
CODE_TIMEOUT, MAX_FEEDBACK_CHARS, SDPO_FEEDBACK_RESERVE = 5, 600, 256
if any(TRAIN_LIMITS[d] < 1 or VAL_LIMITS[d] < 1 for d in SELECTED_DOMAINS):
    raise ValueError("Each selected domain needs positive train/validation counts")
ROUNDS = 2 if SMOKE else 1
MAX_TOKENS = 4 if SMOKE else 256  # Larger original-recipe budgets: Qwen 512, LFM 256.
MAX_SEQ_LENGTH = 256 if SMOKE else 2048
SAMPLES = (2 if SMOKE else 4) if ALGORITHM == "grpo" else 1
CHUNK_SIZE, ACCUMULATION = 8, 4
BENCHMARK_LIMIT = 1 if SMOKE else 32  # None = all available official test questions.
if BENCHMARK_LIMIT is not None and BENCHMARK_LIMIT < 1:
    raise ValueError("BENCHMARK_LIMIT must be positive or None")
if TOOL_TEST_LIMIT < 1 or MAX_TOOL_CALLS < 1 or CODE_TIMEOUT < 1:
    raise ValueError("Tool holdout/call and execution limits must be positive")
HP = {"learning_rate": 2e-5, "gradient_checkpointing": True,
      "gradient_accumulation_steps": ACCUMULATION, "logit_chunk_size": CHUNK_SIZE,
      "max_seq_length": MAX_SEQ_LENGTH, "attention_implementation": "sdpa",
      "max_grad_norm": 1.0, "batch_size": 1, "epochs": 1, "divergence": "jsd"}
random.seed(SEED)
torch.manual_seed(SEED)
CONFIG = {"protocol_version": "domains-v2", "algorithm": ALGORITHM, "model": MODEL_ID, "model_revision": MODEL_REVISION,
          "dataset_revisions": DATA_REVISIONS, "seed": SEED, "domains": SELECTED_DOMAINS, "training_sources": {d: TRAIN_SOURCES[d] for d in SELECTED_DOMAINS},
          "train_limits": TRAIN_LIMITS,
          "validation_limits": VAL_LIMITS, "tool_test_limit": TOOL_TEST_LIMIT,
          "max_tool_calls": MAX_TOOL_CALLS, "code_timeout": CODE_TIMEOUT,
          "max_feedback_chars": MAX_FEEDBACK_CHARS, "sdpo_feedback_reserve": SDPO_FEEDBACK_RESERVE, "rounds": ROUNDS, "max_tokens": MAX_TOKENS,
          "max_seq_length": MAX_SEQ_LENGTH, "samples": SAMPLES, "hyperparameters": HP,
          "benchmark_limit": BENCHMARK_LIMIT, "reference_device": str(REFERENCE_DEVICE),
          "smoke": SMOKE, "versions": {n: metadata.version(n) for n in ("torch", "transformers", "datasets")},
          "device": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU"}
run_digest = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:10]
RUN_DIR = RUN_DIR / ("_".join(SELECTED_DOMAINS) + "_" + run_digest)
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2) + "\n")
print("SMOKE CHECK ONLY" if SMOKE else "Pretrained training recipe", "| Outputs:", RUN_DIR)
if DEVICE.type == "cpu" and not SMOKE:
    print("For practical pretrained training in Colab, select Runtime > Change runtime type > GPU.")

## 2. Domain-specific Hugging Face data and formats

All five algorithms can use these normalized tasks. What changes across domains is the reference, output contract, and verifier; the algorithm's loss remains the one derived later in this tutorial.

| Domain | Training data | Validation | Final benchmarks | Domain score |
|---|---|---|---|---|
| Math | [GSM8K](https://huggingface.co/datasets/openai/gsm8k) official `train`, or [SVAMP](https://huggingface.co/datasets/ChilleD/SVAMP) official `train` | Disjoint reserved training questions | Both official `test` splits | Numeric exact match |
| Coding | [MBPP](https://huggingface.co/datasets/google-research-datasets/mbpp), `full/train` | MBPP `full/validation` | MBPP `full/test`; [HumanEval](https://huggingface.co/datasets/openai/openai_humaneval) `test` | One greedy program passes all provided tests |
| Tool use | [Hermes function calling](https://huggingface.co/datasets/NousResearch/hermes-function-calling-v1), `func-calling-singleturn.json`; optional [xLAM 60k](https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k) | Reserved source tasks | Independent reserved source test + [BFCL](https://huggingface.co/datasets/gorilla-llm/Berkeley-Function-Calling-Leaderboard) `BFCL_v3_simple.json` | Correct tool name(s) and structured arguments |

Math preserves the original sources: GSM8K has 7,473 official training / 1,319 test rows and a worked `answer` ending in `####`; SVAMP has 700 train / 300 test rows with `Body`, `Question`, `Equation`, and `Answer`. Only source tasks selected in `TRAIN_SOURCES` contribute fitting data; both math test sets evaluate transfer when math is selected.

MBPP `full` has separate training, evaluation, test, and prompt splits. The `prompt` split is not used. In this HF release the 90-question evaluation partition is named `validation`. Raw fields are `task_id`, `text`, `code`, `test_list`, `test_setup_code`, and `challenge_test_list`. We use `code` as the private reference and `test_list` as the verifier; challenge tests are not part of this protocol. The public student prompt contains the task and required function signatures, never the reference body or expected test outputs. Signature extraction from the reference specifies the API only. Class-based tasks may require the class described in the task as well. Training references that fail the notebook's execution checks are rejected before sampling; final tests are not filtered by whether the candidate passes.

HumanEval has 164 official test problems, each with `task_id`, a function/docstring `prompt`, `canonical_solution`, `test`, and `entry_point`. Its adapter asks for a **complete implementation** including imports/signature, then executes the hidden `check(entry_point)` tests. It never puts the canonical body or hidden tests in the student prompt. This instruction-style, constrained execution score is an adaptation of HumanEval, not a claim to reproduce a published completion-style leaderboard setting.

Hermes raw rows contain `id`, `tools` (often a JSON string), and ShareGPT `conversations`, with `from` / `value` turns. Only exactly `system → human → gpt` rows are accepted. References contain one or more `<tool_call>{name, arguments}</tool_call>` objects. The adapter unwraps the tool definitions and converts reference calls to a strict JSON array. xLAM has `query`, `tools`, and `answers`; JSON strings and Python literals in **source** fields are normalized, while model outputs must be valid JSON. Both sources supply demonstrations, not live API implementations. Invalid references, unsupported schemas, and actions longer than `MAX_TOOL_CALLS` are rejected and counted. Hermes includes malformed references in this release; this starter deliberately uses only the subset that satisfies the declared call schema.

BFCL is downloaded as question and `possible_answer/` JSONL files with `hf_hub_download`, following its card's instruction not to use `load_dataset`. Questions contain `id`, `question`, and `function`; answer rows have candidate values in `ground_truth`. This notebook handles the single-turn **simple Python** category and uses its own JSON argument matcher with those allowed alternatives, including nested object/array labels. Unique unqualified label names are mapped to the public qualified tool name; the BFCL `any` parameter type is represented by an unconstrained JSON value schema. It reports `BFCL simple JSON-argument match (adapted)`. It does not run BFCL's full AST/language normalization or execution/multi-turn harness and cannot claim an official BFCL aggregate score. A missing optional argument is accepted only when the label explicitly allows omission; tool schemas still require mandatory parameters.

Every source becomes UTF-8 JSONL with a common schema:

| Field | Type | Purpose |
|---|---|---|
| `id`, `dataset_name`, `dataset`, `split` | strings | Stable source identity and split provenance |
| `domain` | `math`, `coding`, `tool_use` | Dispatch to the correct output contract and verifier |
| `task_key` | SHA256 string | Whitespace/case-normalized task text for exact overlap removal |
| `prompt` | string | Only public instructions, interface, and available tool schemas |
| `answer` | string | Numeric answer, `all tests pass`, or a call-array summary for inspection |
| `reference` | string | Worked solution, complete reference program, or canonical call array; private during evaluation |
| `verifier` | object | Numeric check, private tests/setup, or tool schemas + expected calls/candidate labels |

Coding verifier: `{"kind":"python_tests","setup":"","tests":"assert add(2,3)==5"}`. Tool verifier: `{"kind":"tool_calls","tools":[...],"calls":[{"name":"add","arguments":{"a":2,"b":3}}]}`. Numeric verifier: `{"kind":"numeric"}`, using `answer`. These fields stay attached to rollouts so mixed-domain batches do not accidentally use a math extractor on code or tool arguments. The next cell shows synthetic raw/normalized examples for every domain; preparation later downloads the selected real sources.

### Embedded normalization, verifier, and JSONL helpers

These visible functions implement the schema and exact domain verifier used by every stage. Stable IDs are assigned before shuffling. Only `load_examples` calls Hugging Face; all transformations and reward logic are defined here. Data-source revisions and partition hashes are saved beside the training artifacts.

In [ ]:
def read_jsonl(path: str | Path) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    with Path(path).open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON on {path}:{line_number}") from exc
    return records


def write_jsonl(path: str | Path, records: Iterable[dict[str, Any]]) -> None:
    output = Path(path)
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")


def shuffled_batches(
    items: Sequence[Any], batch_size: int, seed: int
) -> Iterator[list[Any]]:
    if batch_size < 1:
        raise ValueError("batch size must be positive")
    order = list(range(len(items)))
    random.Random(seed).shuffle(order)
    for start in range(0, len(order), batch_size):
        yield [items[index] for index in order[start : start + batch_size]]

_BOXED = re.compile(r"\\boxed\{([^{}]+)\}")


_HASH_ANSWER = re.compile(r"####\s*([^\n]+)")


_FINAL_ANSWER = re.compile(r"final answer\s*(?:is|:)?\s*([^\n]+)", re.IGNORECASE)


_NUMBER = re.compile(
    r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?(?:/[-+]?\d+(?:\.\d+)?)?"
)


def _numeric_fraction(text: str) -> Fraction:
    value = Decimal(text)
    if not value.is_finite():
        raise ValueError("nonfinite answer")
    digits = value.as_tuple()
    if len(digits.digits) > 1000 or abs(digits.exponent) > 1000:
        raise ValueError("numeric answer is too large to normalize")
    return Fraction(value)


def normalize_answer(answer: Any) -> str:
    text = str(answer).strip().lower().replace(",", "")
    text = re.sub(r"\s+", " ", text)
    text = text.rstrip(". \n")
    try:
        if "/" in text:
            numerator, denominator = text.split("/")
            value = _numeric_fraction(numerator) / _numeric_fraction(denominator)
        else:
            value = _numeric_fraction(text)
        return str(value)
    except (InvalidOperation, ValueError, ZeroDivisionError, OverflowError):
        return text


def extract_final_answer(text: str) -> str:
    """Extract common math-answer formats, falling back to the final number or line."""
    # Ignore private reasoning when the model emits a thinking block.
    if "</think>" in text:
        text = text.rsplit("</think>", 1)[-1]
    elif "<think>" in text:
        return ""
    text = text.replace(",", "")
    boxed = _BOXED.findall(text)
    if boxed:
        return normalize_answer(boxed[-1])
    hashes = _HASH_ANSWER.findall(text)
    if hashes:
        return normalize_answer(hashes[-1])
    finals = _FINAL_ANSWER.findall(text)
    if finals:
        final = finals[-1].strip()
        number = _NUMBER.match(final)
        return normalize_answer(number.group() if number else final)
    numbers = _NUMBER.findall(text)
    if numbers:
        return normalize_answer(numbers[-1])
    nonempty_lines = [line.strip() for line in text.splitlines() if line.strip()]
    return normalize_answer(nonempty_lines[-1] if nonempty_lines else text)


def exact_match_reward(completion: str, expected_answer: Any) -> float:
    return float(extract_final_answer(completion) == normalize_answer(expected_answer))


def environment_feedback(completion: str, expected_answer: Any) -> str:
    predicted = extract_final_answer(completion)
    expected = normalize_answer(expected_answer)
    if predicted == expected:
        return (
            "The answer passed the verifier. Preserve the correct reasoning and result."
        )
    return (
        f"The answer failed the verifier. It ended with {predicted!r}, but the verified "
        f"answer is {expected!r}. Identify the earliest mistake and reason toward the verified answer."
    )

DATASETS = {
    "gsm8k": ("openai/gsm8k", "main", "math"),
    "svamp": ("ChilleD/SVAMP", "default", "math"),
    "mbpp": ("google-research-datasets/mbpp", "full", "coding"),
    "humaneval": ("openai/openai_humaneval", None, "coding"),
    "hermes": ("NousResearch/hermes-function-calling-v1", "func-calling-singleturn.json", "tool_use"),
    "xlam": ("Salesforce/xlam-function-calling-60k", "xlam_function_calling_60k.json", "tool_use"),
    "bfcl_simple": ("gorilla-llm/Berkeley-Function-Calling-Leaderboard", "BFCL_v3_simple.json", "tool_use"),
}
ANSWER_INSTRUCTION = "Show concise reasoning. End with 'Final answer: <number>'."
TOOL_INSTRUCTION = 'Return only a JSON array of calls: [{"name":"tool_name","arguments":{...}}]. No prose.'
CODE_INSTRUCTION = "Return only complete Python code, preserving the requested interface. No prose."
METRIC_NAMES = {"math": "numeric exact match", "coding": "constrained execution pass@1",
                "tool_use": "structured call accuracy"}


def final_text(text):
    if "</think>" in text:
        text = text.rsplit("</think>", 1)[-1]
    elif "<think>" in text:
        return ""
    text = text.strip()
    fenced = re.fullmatch(r"```(?:python|py|json)?\s*\n(.*?)\n?```", text, re.DOTALL)
    return fenced.group(1).strip("\n") if fenced else text


def strict_json(text):
    def pairs(items):
        result = {}
        for key, value in items:
            if key in result:
                raise ValueError("duplicate JSON key")
            result[key] = value
        return result
    def bad_constant(value):
        raise ValueError(f"nonfinite JSON number: {value}")
    return json.loads(text, object_pairs_hook=pairs, parse_constant=bad_constant)


def source_object(value):
    # Some source references use Python literals. Generated calls must be strict JSON.
    if not isinstance(value, str):
        return value
    try:
        return strict_json(value)
    except ValueError:
        return ast.literal_eval(value)


def normalize_schema(schema):
    aliases = {"dict": "object", "int": "integer", "float": "number", "bool": "boolean",
               "list": "array", "tuple": "array", "str": "string"}
    result = dict(schema)
    if "type" in result and isinstance(result["type"], str):
        result["type"] = aliases.get(result["type"], result["type"])
    if result.get("type") == "any":
        result.pop("type")
    if "properties" in result:
        result["properties"] = {k: normalize_schema(v) for k, v in result["properties"].items()}
        result.setdefault("additionalProperties", False)
    if isinstance(result.get("items"), dict):
        result["items"] = normalize_schema(result["items"])
    if isinstance(result.get("additionalProperties"), dict):
        result["additionalProperties"] = normalize_schema(result["additionalProperties"])
    return result


def normalize_tools(value):
    tools = []
    for item in source_object(value):
        item = dict(item.get("function", item))
        schema = item.get("parameters", {"type": "object", "properties": {}})
        if "type" not in schema and "properties" not in schema:
            schema = {"type": "object", "properties": schema, "required": list(schema)}
        item["parameters"] = normalize_schema(schema)
        Draft7Validator.check_schema(item["parameters"])
        tools.append(item)
    if len({t["name"] for t in tools}) != len(tools):
        raise ValueError("duplicate tool names")
    return tools


def validate_calls(calls, tools):
    if not isinstance(calls, list):
        raise TypeError("expected a JSON array of calls")
    definitions = {t["name"]: t for t in tools}
    for call in calls:
        if not isinstance(call, dict) or set(call) != {"name", "arguments"}:
            raise ValueError("each call needs exactly name and arguments")
        if not isinstance(call["name"], str) or call["name"] not in definitions:
            raise ValueError("unknown tool name")
        if not isinstance(call["arguments"], dict):
            raise TypeError("arguments must be an object")
        Draft7Validator(definitions[call["name"]]["parameters"]).validate(call["arguments"])


def same_value(left, right):
    # Dictionary order is irrelevant. Array order and boolean-vs-number types matter.
    if isinstance(left, bool) or isinstance(right, bool):
        return type(left) is type(right) and left == right
    if isinstance(left, (int, float)) and isinstance(right, (int, float)):
        return left == right
    if type(left) is not type(right):
        return False
    if isinstance(left, dict):
        return left.keys() == right.keys() and all(same_value(left[k], right[k]) for k in left)
    if isinstance(left, list):
        return len(left) == len(right) and all(same_value(a, b) for a, b in zip(left, right))
    return left == right


def bfcl_value_matches(actual, expected, schema):
    if isinstance(expected, dict) and schema.get("type") == "object":
        return isinstance(actual, dict) and bfcl_arguments_match(actual, expected, schema)
    if isinstance(expected, list) and schema.get("type") == "array":
        return (isinstance(actual, list) and len(actual) == len(expected) and
                all(bfcl_value_matches(a, b, schema.get("items", {})) for a, b in zip(actual, expected)))
    return same_value(actual, expected)


def bfcl_arguments_match(arguments, target, schema):
    if arguments.keys() - target.keys():
        return False
    properties, required = schema.get("properties", {}), schema.get("required", [])
    return all(any(bfcl_value_matches(arguments[k], v, properties.get(k, {})) for v in choices)
               if k in arguments else k not in required and "" in choices
               for k, choices in target.items())


def bfcl_canonical_value(expected, schema):
    if isinstance(expected, dict) and schema.get("type") == "object":
        return bfcl_canonical_arguments(expected, schema)
    if isinstance(expected, list) and schema.get("type") == "array":
        return [bfcl_canonical_value(v, schema.get("items", {})) for v in expected]
    return expected


def bfcl_canonical_arguments(target, schema):
    arguments = {}
    for key, choices in target.items():
        property_schema = schema.get("properties", {}).get(key, {})
        for choice in choices:
            if choice == "":
                continue  # Empty string in these labels denotes optional omission.
            value = bfcl_canonical_value(choice, property_schema)
            if Draft7Validator(property_schema).is_valid(value):
                arguments[key] = value
                break
        else:
            if key in schema.get("required", []) or "" not in choices:
                raise ValueError("BFCL labels cannot satisfy the declared schema")
    Draft7Validator(schema).validate(arguments)
    return arguments


# Visible, self-contained worker: no generated Python runs in the notebook process.
CODE_WORKER = r'''
import ast, builtins, contextlib, importlib, io, json, resource, sys, traceback
allowed = {name: importlib.import_module(name) for name in
           ("math", "cmath", "re", "itertools", "functools", "collections", "heapq",
            "bisect", "string", "typing", "statistics", "fractions", "decimal", "random")}
payload = json.loads(sys.stdin.read())
resource.setrlimit(resource.RLIMIT_CPU, (2, 2))
resource.setrlimit(resource.RLIMIT_AS, (512 * 1024**2, 512 * 1024**2))
resource.setrlimit(resource.RLIMIT_FSIZE, (0, 0))
resource.setrlimit(resource.RLIMIT_NOFILE, (32, 32))
resource.setrlimit(resource.RLIMIT_NPROC, (1, 1))
blocked = {"open", "exec", "eval", "compile", "input", "breakpoint", "help", "globals",
           "locals", "vars", "dir", "getattr", "setattr", "delattr", "__import__",
           "exit", "quit", "copyright", "license", "credits"}
safe = {k: v for k, v in vars(builtins).items() if k not in blocked}
def imports(name, globals=None, locals=None, fromlist=(), level=0):
    if level or name not in allowed or any(x.startswith("_") for x in (fromlist or ())):
        raise ValueError("unsupported import")
    return allowed[name]
safe["__import__"] = imports
scope = {"__builtins__": safe, "__name__": "candidate"}
try:
    program = "\n".join((payload.get("setup", ""), payload["code"], payload["tests"]))
    tree = ast.parse(program)
    for node in ast.walk(tree):
        if isinstance(node, ast.Attribute) and node.attr.startswith("_"):
            raise ValueError("private attributes are unsupported")
        if isinstance(node, ast.Name) and (node.id in blocked or node.id.startswith("__")):
            raise ValueError("restricted builtin or private name")
        if isinstance(node, ast.Import) and any(x.name not in allowed for x in node.names):
            raise ValueError("unsupported import")
        if isinstance(node, ast.ImportFrom) and (node.level or node.module not in allowed):
            raise ValueError("unsupported import")
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        exec(compile(tree, "<candidate-and-tests>", "exec"), scope)
    result = {"passed": True, "diagnostic": "All provided tests passed."}
except BaseException as exc:
    frames = [f for f in traceback.extract_tb(exc.__traceback__) if f.filename == "<candidate-and-tests>"]
    line = program.splitlines()[frames[-1].lineno - 1] if frames else ""
    result = {"passed": False, "diagnostic": (type(exc).__name__ + ": " + str(exc) + " | " + line)[:350]}
print(json.dumps(result))
'''


def check_code(completion, verifier):
    payload = {"code": final_text(completion), "setup": verifier.get("setup", ""),
               "tests": verifier["tests"]}
    try:
        with tempfile.TemporaryDirectory(prefix="rl_code_") as work:
            process = subprocess.run([sys.executable, "-I", "-c", CODE_WORKER],
                input=json.dumps(payload), text=True, capture_output=True, cwd=work,
                env={"PATH": os.defpath}, timeout=CODE_TIMEOUT, check=False)
        if process.returncode:
            return {"passed": False, "diagnostic": f"Worker terminated: exit {process.returncode}"}
        try:
            return json.loads(process.stdout)
        except (ValueError, TypeError):
            return {"passed": False, "diagnostic": "Worker returned no valid result"}
    except subprocess.TimeoutExpired:
        return {"passed": False, "diagnostic": "Execution timeout"}


def verify_completion(completion, row):
    domain, spec = row["domain"], row["verifier"]
    if domain == "math":
        reward = exact_match_reward(completion, row["answer"])
        return {"reward": reward, "predicted": extract_final_answer(completion),
                "feedback": environment_feedback(completion, row["answer"])}
    if domain == "coding":
        result = check_code(completion, spec)
        return {"reward": float(result["passed"]), "predicted": final_text(completion),
                "feedback": "Python test result for this attempt: " + result["diagnostic"]}
    calls, json_valid, schema_valid = None, False, False
    try:
        calls = strict_json(final_text(completion))
        json_valid = isinstance(calls, list)
        validate_calls(calls, spec["tools"])
        schema_valid = True
        if spec.get("kind") == "bfcl_candidates":
            match = len(calls) == 1 and calls[0]["name"] == spec["name"]
            definition = next(t for t in spec["tools"] if t["name"] == spec["name"])
            match = match and bfcl_arguments_match(calls[0]["arguments"], spec["allowed_arguments"], definition["parameters"])
        else:
            # Preserve call order; these recipes do not execute dependencies between calls.
            match = same_value(calls, spec["calls"])
        diagnostic = "Correct tool names and arguments." if match else "Call names, order, or arguments differ from the reference."
    except (ValueError, TypeError, KeyError, ValidationError) as exc:
        match, diagnostic = False, type(exc).__name__ + ": " + str(exc)[:250]
    return {"reward": float(match), "predicted": calls, "json_valid": float(json_valid),
            "schema_valid": float(schema_valid), "feedback": "Tool verifier for this attempt: " + diagnostic}


def rewrite_prompt(row):
    instruction = {"math": ANSWER_INSTRUCTION, "coding": CODE_INSTRUCTION,
                   "tool_use": TOOL_INSTRUCTION}[row["domain"]]
    return (f"Rewrite the reference in your own style, preserving its behavior and correctness.\n"
            f"{instruction}\n\nTask:\n{row['prompt']}\n\nReference:\n{row['reference']}")


def normalize_example(name, row, split, index):
    domain = DATASETS[name][2]
    identity = row.get("task_id", row.get("id", row.get("ID", index)))
    result = {"id": f"{name}:{split}:{identity}", "domain": domain,
              "dataset": DATASETS[name][0], "dataset_name": name, "split": split}
    if domain == "math":
        if name == "gsm8k":
            question, reference = row["question"].strip(), row["answer"].strip()
            if "####" not in reference:
                raise ValueError("GSM8K reference lacks ####")
            answer = extract_final_answer(reference)
        else:
            question = f"{row['Body'].strip()} {row['Question'].strip()}"
            answer = normalize_answer(row["Answer"])
            reference = f"Compute {row['Equation']}. Final answer: {answer}"
        result.update(prompt=f"{question}\n\n{ANSWER_INSTRUCTION}", answer=answer,
                      reference=reference, verifier={"kind": "numeric"})
    elif domain == "coding":
        if name == "mbpp":
            question, reference = row["text"], row["code"]
            definitions = [n for n in ast.parse(reference).body if isinstance(n, ast.FunctionDef)]
            interface = "\n".join(f"def {n.name}({ast.unparse(n.args)}): ..." for n in definitions)
            prompt = f"{question}\nRequired interface:\n{interface}\n\n{CODE_INSTRUCTION}"
            verifier = {"kind": "python_tests", "setup": row.get("test_setup_code", ""),
                        "tests": "\n".join(row["test_list"])}
        else:
            question = row["prompt"]
            reference = row["prompt"] + row["canonical_solution"]
            prompt = f"Complete this Python specification. Return the entire function and needed imports:\n{question}\n{CODE_INSTRUCTION}"
            verifier = {"kind": "python_tests", "tests": row["test"] + f"\ncheck({row['entry_point']})"}
        result.update(prompt=prompt, reference=reference, answer="all tests pass", verifier=verifier)
    else:
        if name == "hermes":
            conversation = row["conversations"]
            if [c["from"] for c in conversation] != ["system", "human", "gpt"]:
                raise ValueError("requires a single user turn and one assistant action")
            question, tools = conversation[1]["value"], normalize_tools(row["tools"])
            calls = [source_object(text) for text in re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", conversation[2]["value"], re.DOTALL)]
        elif name == "xlam":
            question, tools = row["query"], normalize_tools(row["tools"])
            calls = source_object(row["answers"])
            for call in calls:
                call["arguments"] = source_object(call["arguments"])
        else:
            if len(row["question"]) != 1 or len(row["question"][0]) != 1:
                raise ValueError("BFCL adapter supports single-turn simple calls only")
            question, tools = row["question"][0][0]["content"], normalize_tools(row["function"])
            truth = row["ground_truth"]
            if len(truth) != 1 or len(truth[0]) != 1:
                raise ValueError("requires exactly one BFCL target function")
            function, allowed = next(iter(truth[0].items()))
            names = [t["name"] for t in tools]
            if function not in names:
                qualified = [name for name in names if name.rsplit(".", 1)[-1] == function]
                if len(qualified) != 1:
                    raise ValueError("BFCL target does not identify an available tool")
                function = qualified[0]
            definition = next(t for t in tools if t["name"] == function)
            calls = [{"name": function, "arguments": bfcl_canonical_arguments(allowed, definition["parameters"])}]
        if not calls or len(calls) > MAX_TOOL_CALLS:
            raise ValueError("empty calls or too many calls for this starter recipe")
        if name != "bfcl_simple":
            validate_calls(calls, tools)
        prompt = f"Request:\n{question}\n\nAvailable tools:\n{json.dumps(tools, separators=(',', ':'))}\n\n{TOOL_INSTRUCTION}"
        verifier = {"kind": "tool_calls", "tools": tools, "calls": calls}
        if name == "bfcl_simple":
            verifier = {"kind": "bfcl_candidates", "tools": tools, "name": function, "allowed_arguments": allowed}
        result.update(prompt=prompt, reference=json.dumps(calls, separators=(",", ":")),
                      answer=json.dumps(calls), verifier=verifier)
    if not question.strip() or not result["reference"].strip():
        raise ValueError("empty task or reference")
    # Task-text overlap checks work even if dataset-specific output instructions differ.
    result["task_key"] = hashlib.sha256(re.sub(r"\s+", " ", question).strip().lower().encode()).hexdigest()
    return result


def load_examples(name, split, revision="main"):
    from datasets import load_dataset
    from huggingface_hub import hf_hub_download
    path, config, domain = DATASETS[name]
    if name == "bfcl_simple":
        questions = read_jsonl(hf_hub_download(path, config, repo_type="dataset", revision=revision))
        answers = read_jsonl(hf_hub_download(path, "possible_answer/" + config, repo_type="dataset", revision=revision))
        labels = {r["id"]: r["ground_truth"] for r in answers}
        raw = [{**r, "ground_truth": labels[r["id"]]} for r in questions]
    elif domain == "tool_use":
        try:
            filename = hf_hub_download(path, config, repo_type="dataset", revision=revision,
                                      token=os.environ.get("HF_TOKEN"))
        except Exception as exc:
            if name == "xlam":
                raise RuntimeError("xLAM requires accepting its HF dataset terms and setting HF_TOKEN. Use hermes for an ungated run.") from exc
            raise
        raw = json.loads(Path(filename).read_text())
    else:
        raw = load_dataset(path, config, split=split, revision=revision)
    records, rejected = [], Counter()
    for index, row in enumerate(raw):
        try:
            records.append(normalize_example(name, dict(row), split, index))
        except (ValueError, TypeError, KeyError, SyntaxError, ValidationError, SchemaError) as exc:
            rejected[type(exc).__name__] += 1
    return records, {"raw_count": len(raw), "normalized_count": len(records),
                     "normalization_rejections": dict(rejected), "source": path, "revision": revision}


In [ ]:
synthetic_rows = {
    "gsm8k": {"question": "There are 2 red and 3 blue balls. How many balls?", "answer": "Add 2 + 3. #### 5"},
    "mbpp": {"task_id": "demo", "text": "Add a and b.", "code": "def add(a, b):\n    return a + b",
             "test_list": ["assert add(2, 3) == 5"]},
    "xlam": {"id": "demo", "query": "Add 2 and 3 using the add tool.",
             "tools": [{"name": "add", "parameters": {"type": "object", "properties": {
                 "a": {"type": "integer"}, "b": {"type": "integer"}}, "required": ["a", "b"]}}],
             "answers": [{"name": "add", "arguments": {"a": 2, "b": 3}}]},
}
demo_examples = {DATASETS[name][2]: normalize_example(name, raw, "demo", 0)
                 for name, raw in synthetic_rows.items()}
for domain, row in demo_examples.items():
    print(domain, "raw:", json.dumps(synthetic_rows[row["dataset_name"]], indent=2))
    print("Normalized:", json.dumps(row, indent=2))


### Partitioning, filtering, and domain-wise artifacts

The notebook loads only selected training sources and the final benchmarks for selected domains. It reserves tool-test questions first, gathers every final-test task key, then excludes matching questions from all fitting pools. Training and validation are disjoint. Math/tool validation comes from remaining training pools; coding validation comes from MBPP's separate `validation` split. An entire tool action is a task, so calls from one example are never split across partitions.

Within each fitting domain, references are normalized, duplicate training task text is removed, context limits are checked, and training programs are checked against their provided tests. A fixed seed samples `TRAIN_LIMITS[domain]` fitting tasks and `VAL_LIMITS[domain]` validation tasks. Too few eligible rows raises an actionable error. These are question-level tool holdouts; shared tools across partitions are allowed. BFCL provides an additional source of tool requests, but this protocol does not guarantee unseen tool families or removal of semantic paraphrases/pretraining contamination.

`train_math.jsonl`, `train_coding.jsonl`, and `train_tool_use.jsonl` contain only selected domains; matching `validation_<domain>.jsonl` files preserve per-domain validation. `train.jsonl` is the shuffled mixture used by the trainer. Final `<dataset>_test.jsonl` files remain separate. `data_protocol.json` records IDs, requested source revisions, source/normalization/filter counts, source split provenance, and SHA256 of every prepared JSONL. The reserved tool test is explicitly labeled `reserved train`, never presented as an official publisher test split.

Both checkpoints are evaluated on exactly the same prepared test IDs. Tests that exceed the shared context limit are excluded **before either model is evaluated** and their number is reported. This changes benchmark coverage: even `BENCHMARK_LIMIT=None` means all prepared tasks, not necessarily all original tasks. No tests are removed based on a trained model's outcome, reference correctness, or generated response length.

The preparation function is defined here and called after tokenizer/model helpers are available. It uses the seed tokenizer for context checks; saved policies retain that tokenizer.


Official benchmark rows retain their unique source IDs even when two problems share task wording. Deduplication applies to fitting/source-holdout pools; retaining published benchmark rows preserves the original denominator when every row normalizes and fits the context limit.

In [ ]:
def smoke_examples(name, split):
    domain = DATASETS[name][2]
    rows = []
    for i in (range(1, 6) if split == "train" else range(8, 10)):
        if name == "gsm8k":
            raw = {"question": f"Smoke {name}: {i} + 3 ?", "answer": f"Add 3. #### {i + 3}"}
        elif name == "svamp":
            raw = {"ID": i, "Body": f"Smoke {name}: {i} balls.", "Question": "Add 3. Total?",
                   "Equation": f"{i}+3", "Answer": i + 3}
        elif domain == "coding":
            function = f"add_{name}_{split}_{i}"
            code = f"def {function}(n):\n    return n + {i}\n"
            if name == "mbpp":
                raw = {"task_id": f"{split}_{i}", "text": f"Add {i} to n in {function}.", "code": code,
                       "test_list": [f"assert {function}(2) == {i + 2}"]}
            else:
                raw = {"task_id": f"{split}_{i}", "prompt": f'def {function}(n):\n    """Add {i} to n."""\n',
                       "canonical_solution": f"    return n + {i}\n", "entry_point": function,
                       "test": f"def check(candidate):\n    assert candidate(2) == {i + 2}"}
        else:
            query = f"Smoke {name} {split} {i}: call add with n={i}."
            tools = [{"name": "add", "parameters": {"type": "object",
                      "properties": {"n": {"type": "integer"}}, "required": ["n"]}}]
            calls = [{"name": "add", "arguments": {"n": i}}]
            if name == "hermes":
                raw = {"id": i, "tools": tools, "conversations": [{"from": "system", "value": "Tools"},
                       {"from": "human", "value": query}, {"from": "gpt", "value": f"<tool_call>{json.dumps(calls[0])}</tool_call>"}]}
            elif name == "xlam":
                raw = {"id": i, "query": query, "tools": tools, "answers": calls}
            else:
                raw = {"id": i, "question": [[{"role": "user", "content": query}]], "function": tools,
                       "ground_truth": [{"add": {"n": [i]}}]}
        rows.append(normalize_example(name, raw, split, i))
    return rows


def unique_records(rows, excluded=()):
    seen, kept = set(excluded), []
    for row in rows:
        if row["task_key"] not in seen:
            seen.add(row["task_key"])
            kept.append(row)
    return kept


def fits_context(row, tokenizer, training=False):
    prompt_length = len(chat_prompt_ids(tokenizer, row["prompt"]))
    if prompt_length + MAX_TOKENS > MAX_SEQ_LENGTH:
        return False
    if not training:
        return True
    if ALGORITHM == "sdft":
        target = len(tokenizer.encode(row["reference"], add_special_tokens=False)) + 1
        return (prompt_length + target <= MAX_SEQ_LENGTH and
                len(chat_prompt_ids(tokenizer, rewrite_prompt(row))) + MAX_TOKENS <= MAX_SEQ_LENGTH)
    if ALGORITHM == "opsd":
        return len(chat_prompt_ids(tokenizer, opsd_teacher_prompt(row), OPSD_SYSTEM)) + MAX_TOKENS <= MAX_SEQ_LENGTH
    if ALGORITHM == "sdpo":
        # Actual feedback is checked again after generation; this reserves typical space.
        return prompt_length + MAX_TOKENS + (32 if SMOKE else SDPO_FEEDBACK_RESERVE) <= MAX_SEQ_LENGTH
    return True


def prepare_data():
    tokenizer = load_tokenizer(MODEL)
    loaded, manifests, paths = {}, {}, {}
    def get(name, split):
        key = f"{name}/{split}"
        if key not in loaded:
            if SMOKE:
                rows = smoke_examples(name, split)
                detail = {"source": "synthetic smoke fixture", "raw_count": len(rows),
                          "normalized_count": len(rows), "revision": "SMOKE"}
            else:
                rows, detail = load_examples(name, split, DATA_REVISIONS[name])
            loaded[key], manifests[key] = rows, detail
        return loaded[key]

    benchmark_sources = {"math": ["gsm8k", "svamp"], "coding": ["mbpp", "humaneval"],
                         "tool_use": ["bfcl_simple"]}
    tests, pools, official_validation = {}, {d: [] for d in SELECTED_DOMAINS}, {}
    for domain in SELECTED_DOMAINS:
        for name in benchmark_sources[domain]:
            tests[name] = list(get(name, "test"))
            if len({r["id"] for r in tests[name]}) != len(tests[name]):
                raise ValueError(f"Duplicate benchmark IDs in {name}")
        for name in TRAIN_SOURCES[domain]:
            if name not in DATASETS or DATASETS[name][2] != domain or name in ("humaneval", "bfcl_simple"):
                raise ValueError(f"{name} is not an applicable {domain} training source")
            rows = unique_records(get(name, "train"))
            random.Random(SEED).shuffle(rows)
            if domain == "tool_use":
                count = 1 if SMOKE else TOOL_TEST_LIMIT
                tests[f"{name}_heldout"] = [{**r, "split": "test", "source_split": "train"} for r in rows[:count]]
                rows = rows[count:]
            pools[domain].extend(rows)
        if domain == "coding":
            official_validation[domain] = get("mbpp", "validation")

    # Reserve every selected-domain test question before selecting any fitting data.
    heldout_keys = {r["task_key"] for rows in tests.values() for r in rows}
    train, validation, filtering = [], [], {}
    for domain, pool in pools.items():
        eligible, reasons = [], Counter()
        for row in unique_records(pool, heldout_keys):
            if not fits_context(row, tokenizer, training=True):
                reasons["training_context_limit"] += 1
                continue
            if domain == "coding" and not check_code(row["reference"], row["verifier"])["passed"]:
                reasons["reference_failed_constrained_tests"] += 1
                continue
            eligible.append(row)
        random.Random(SEED).shuffle(eligible)
        val_count, train_count = VAL_LIMITS[domain], TRAIN_LIMITS[domain]
        if domain in official_validation:
            candidates = [r for r in unique_records(official_validation[domain], heldout_keys)
                          if fits_context(r, tokenizer)]
            random.Random(SEED).shuffle(candidates)
            chosen_val = candidates[:val_count]
            val_keys = {r["task_key"] for r in chosen_val}
            eligible = [r for r in eligible if r["task_key"] not in val_keys]
        else:
            chosen_val, eligible = eligible[:val_count], eligible[val_count:]
        chosen_train = eligible[:train_count]
        if len(chosen_val) != val_count or len(chosen_train) != train_count:
            raise ValueError(f"Too few eligible {domain} rows: train={len(chosen_train)}, validation={len(chosen_val)}. "
                             "Reduce domain limits/TOOL_TEST_LIMIT or increase MAX_SEQ_LENGTH.")
        train.extend(chosen_train)
        validation.extend({**r, "split": "validation", "source_split": r["split"]} for r in chosen_val)
        filtering[domain] = {"pool_count": len(pool), "dedup_or_test_overlap": len(pool) - len(unique_records(pool, heldout_keys)),
                             "rejections": dict(reasons), "training_count": train_count, "validation_count": val_count}
    random.Random(SEED).shuffle(train)
    fitting_keys = {r["task_key"] for r in train + validation}
    assert not fitting_keys.intersection(heldout_keys), "Training/validation overlap final tests"
    assert not {r["task_key"] for r in train}.intersection(r["task_key"] for r in validation)
    for label, rows in {"train": train, "validation": validation}.items():
        paths[label] = RUN_DIR / f"{label}.jsonl"
        write_jsonl(paths[label], rows)
    for domain in SELECTED_DOMAINS:
        write_jsonl(RUN_DIR / f"train_{domain}.jsonl", [r for r in train if r["domain"] == domain])
        write_jsonl(RUN_DIR / f"validation_{domain}.jsonl", [r for r in validation if r["domain"] == domain])
    for name, rows in list(tests.items()):
        raw_count = len(rows)
        tests[name] = [r for r in rows if fits_context(r, tokenizer)]
        if not tests[name]:
            raise ValueError(f"No benchmark rows fit the context budget for {name}")
        random.Random(SEED).shuffle(tests[name])
        paths[name] = RUN_DIR / f"{name}_test.jsonl"
        write_jsonl(paths[name], tests[name])
        manifests[f"{name}/prepared_test"] = {"available_before_context_filter": raw_count,
            "available_after_context_filter": len(tests[name]), "context_excluded": raw_count - len(tests[name]),
            "source_normalization_excluded": (manifests.get(f"{name}/test", {}).get("raw_count", raw_count)
                                             - manifests.get(f"{name}/test", {}).get("normalized_count", raw_count)),
            "source_split": "reserved train" if name.endswith("_heldout") else "official test"}
    protocol = {"seed": SEED, "domains": SELECTED_DOMAINS, "sources": manifests, "filtering": filtering,
                "training_ids": [r["id"] for r in train], "validation_ids": [r["id"] for r in validation],
                "test_ids": {name: [r["id"] for r in rows] for name, rows in tests.items()},
                "prepared_sha256": {key: hashlib.sha256(path.read_bytes()).hexdigest() for key, path in paths.items()},
                "benchmark_protocol": "one greedy full response; domain verifier; adapted HumanEval/BFCL scores"}
    (RUN_DIR / "data_protocol.json").write_text(json.dumps(protocol, indent=2) + "\n")
    return train, validation, tests, paths


## 3. Domain verifiers, rewards, and fresh feedback

Let $V_d(x,y)$ be the domain verifier. Math extracts/normalizes the final number; coding executes the generated program with private tests; tools parse a strict JSON call array, validate parameter schemas, and compare structured arguments. All three return $R=\mathbf{1}\{V_d(x,y)\text{ passes}\}$. Numeric equality does not verify the reasoning, provided tests do not cover every possible program behavior, and matching a reference API call does not establish real task completion.

Coding runs in a separate `python -I` worker with a temporary working directory, a minimal environment, time/CPU/memory/file/process limits, restricted builtins/imports, and AST checks. The notebook process never executes generated programs. The allowed standard libraries are visible in `CODE_WORKER`; filesystem/network access and private-attribute introspection are unsupported. This is a constrained tutorial evaluator, **not a security sandbox**. Run on a disposable Colab runtime without mounted private files; stronger adversarial evaluation needs an isolated service/container. Rejected code counts as incorrect. The same restrictions apply to seed, trained model, and training-reference checks, and can reduce coverage compared with official unrestricted execution harnesses.

Tool outputs must be a JSON array. Dictionary-key order is ignored, array/call order is preserved, booleans are distinguished from numbers, and duplicate JSON keys / NaN / infinity are rejected. JSON syntax and schema validity are recorded separately from argument correctness. Calls are **checked, not executed**: there are no real network tools, environment observations, or multi-turn agent tasks in these notebooks. Multi-turn APIGen/τ-bench data is omitted because these single-response training loops cannot mask and advance those trajectories correctly.

| Recipe | How these domains supply its learning signal |
|---|---|
| SDFT | Rewrite a correct private reference; accept only if the domain verifier passes and generation is not truncated; otherwise keep the original verified reference |
| SDPO | Generate a new attempt, then use its actual numeric/test/schema/argument diagnostic in the detached self-teacher context; stored unrelated feedback is never reused |
| OPSD | Give a frozen teacher the reference solution/program/call array while the student sees only the public task; match distributions on the same sampled continuation |
| PPO | Use the domain's terminal binary reward with old action log-probabilities, a value head, GAE, and clipping |
| GRPO | Sample a complete group per task, normalize rewards within that task, and clip the policy ratio with the frozen seed reference |

SDFT/SDPO/OPSD do not inherently require a scalar RL reward; these verifiers provide reproducible acceptance, feedback, and evaluation in this tutorial. Small models can receive all-zero code/tool rewards. In GRPO, $A_i=(R_i-\overline R)/(\operatorname{std}(R)+\epsilon)$ is zero when the group's rewards are equal, so there is no reward-learning signal in that group. The rollout cell prints the fraction of groups with reward variance. Use easier tasks or a suitable supervised warm start if that persists; none of these notebooks promises an improvement. SDPO test errors still provide token-level information even when the terminal reward is zero, although sparse assertion errors can be weak teachers.

In [ ]:
for completion, expected in [("Final answer: 5", "5"), ("Final answer: 1/2", "0.5")]:
    assert exact_match_reward(completion, expected) == 1
assert verify_completion("def add(a, b):\n    return a + b", demo_examples["coding"])["reward"] == 1
assert verify_completion("def add(a, b):\n    return a - b", demo_examples["coding"])["reward"] == 0
assert verify_completion('[{"name":"add","arguments":{"b":3,"a":2}}]', demo_examples["tool_use"])["reward"] == 1
assert verify_completion('[{"name":"add","arguments":{"a":2,"b":4}}]', demo_examples["tool_use"])["reward"] == 0
assert verify_completion('[{"name":"add","arguments":{"a":true,"b":3}}]', demo_examples["tool_use"])["reward"] == 0
print("Numeric, executable-code, and structured-tool verifier examples passed.")

assert verify_completion('[{"name":"add","arguments":{"a":2,"a":4,"b":3}}]', demo_examples["tool_use"])["reward"] == 0
assert verify_completion('[{"name":"add","arguments":{"a":NaN,"b":3}}]', demo_examples["tool_use"])["reward"] == 0
assert not check_code("import os\nraise RuntimeError('should not execute')", {"tests": "assert True"})["passed"]
assert not check_code("while True:\n    pass", {"tests": "assert True"})["passed"]
encoded_source = {**synthetic_rows["xlam"], "tools": json.dumps(synthetic_rows["xlam"]["tools"]),
                  "answers": json.dumps(synthetic_rows["xlam"]["answers"])}
encoded_example = normalize_example("xlam", encoded_source, "demo", 1)
assert verify_completion(encoded_example["reference"], encoded_example)["reward"] == 1
nested_schema = {"type": "object", "properties": {"filter": {"type": "object",
                 "properties": {"city": {"type": "string"}}, "required": ["city"]}}, "required": ["filter"]}
nested_labels = {"filter": [{"city": ["London", "LONDON"]}]}
assert bfcl_arguments_match({"filter": {"city": "London"}}, nested_labels, nested_schema)
assert not bfcl_arguments_match({"filter": {"city": "Paris"}}, nested_labels, nested_schema)
assert bfcl_canonical_arguments(nested_labels, nested_schema) == {"filter": {"city": "London"}}
print("Source-string normalization, nested BFCL alternatives, restricted imports, and execution limits checked.")


## 4. Tokens, causal shifting, and masks

Let $x$ denote the ordinary chat prompt and $y=(y_1,\ldots,y_T)$ its completion. An autoregressive policy factorizes as

$$\pi_\theta(y\mid x)=\prod_{t=1}^{T}\pi_\theta(y_t\mid x,y_{<t}).$$

The tokenizer renders the model's system/user chat template and an assistant generation prefix. The trainer concatenates prompt IDs and completion IDs, right-pads sequences, and computes:

| Tensor | Shape | Meaning |
|---|---|---|
| `input_ids` | $[B,S]$ | Prompt followed by completion, then padding |
| `attention_mask` | $[B,S]$ | Real input tokens, including both prompt and response |
| `action_mask` | $[B,S-1]$ | Positions whose next-token prediction belongs to the completion |
| Chosen log probabilities / values | $[B,S-1]$ | Shifted predictions; masked positions do not enter losses |
| Completion hidden states | list of $[T_i,H]$ | Decoder states that predict the sampled completion tokens |

At zero-based logit position $P-1$, the model predicts the **first** completion token when the prompt has length $P$. The action interval is therefore `[P-1 : P-1+T]`, not `[P : P+T]`. Prompt tokens help the model through attention but receive no direct response loss; padding receives neither attention nor loss. The toy integer IDs below make the shift visible without downloading a tokenizer.

In [ ]:
DEFAULT_SYSTEM_PROMPT = (
    "You are a careful assistant. Follow the task-specific output format exactly."
)


def chat_prompt_ids(
    tokenizer: Any,
    user_prompt: str,
    system_prompt: str = DEFAULT_SYSTEM_PROMPT,
    *,
    enable_thinking: bool = False,
) -> list[int]:
    """Render one user turn, with a fallback for base models without a chat template."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    if getattr(tokenizer, "chat_template", None):
        rendered = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=enable_thinking,
        )
        if hasattr(rendered, "keys"):
            rendered = rendered["input_ids"]
        return list(rendered)
    text = f"System: {system_prompt}\nUser: {user_prompt}\nAssistant:"
    return list(tokenizer.encode(text, add_special_tokens=True))


@dataclass
class CausalBatch:
    input_ids: torch.Tensor
    attention_mask: torch.Tensor
    action_mask: torch.Tensor
    prompt_lengths: list[int]
    completion_lengths: list[int]
    old_logprobs: torch.Tensor | None = None

    def to(self, device: torch.device) -> "CausalBatch":
        return CausalBatch(
            input_ids=self.input_ids.to(device),
            attention_mask=self.attention_mask.to(device),
            action_mask=self.action_mask.to(device),
            prompt_lengths=self.prompt_lengths,
            completion_lengths=self.completion_lengths,
            old_logprobs=None
            if self.old_logprobs is None
            else self.old_logprobs.to(device),
        )


def causal_batch(
    prompt_token_ids: Sequence[Sequence[int]],
    completion_token_ids: Sequence[Sequence[int]],
    pad_token_id: int,
    old_token_logprobs: Sequence[Sequence[float]] | None = None,
) -> "CausalBatch":
    """Right-pad prompt+completion sequences and mark only completion predictions."""
    if len(prompt_token_ids) != len(completion_token_ids):
        raise ValueError("prompt and completion batch sizes differ")
    if not prompt_token_ids:
        raise ValueError("cannot build an empty batch")
    if old_token_logprobs is not None and len(old_token_logprobs) != len(
        prompt_token_ids
    ):
        raise ValueError("old log-probability and sample batch sizes differ")

    prompt_lengths = [len(ids) for ids in prompt_token_ids]
    completion_lengths = [len(ids) for ids in completion_token_ids]
    if min(prompt_lengths) < 1 or min(completion_lengths) < 1:
        raise ValueError("every sample needs at least one prompt and completion token")
    max_length = max(p + c for p, c in zip(prompt_lengths, completion_lengths))

    input_ids = torch.full(
        (len(prompt_lengths), max_length), pad_token_id, dtype=torch.long
    )
    attention_mask = torch.zeros_like(input_ids, dtype=torch.bool)
    action_mask = torch.zeros((len(prompt_lengths), max_length - 1), dtype=torch.bool)
    old_logprobs = None
    if old_token_logprobs is not None:
        old_logprobs = torch.zeros(
            (len(prompt_lengths), max_length - 1), dtype=torch.float32
        )

    for row, (prompt_ids, completion_ids) in enumerate(
        zip(prompt_token_ids, completion_token_ids)
    ):
        sequence = list(prompt_ids) + list(completion_ids)
        input_ids[row, : len(sequence)] = torch.tensor(sequence, dtype=torch.long)
        attention_mask[row, : len(sequence)] = True
        # logits[:, position] predicts input_ids[:, position + 1].
        start = len(prompt_ids) - 1
        stop = start + len(completion_ids)
        action_mask[row, start:stop] = True
        if old_logprobs is not None:
            row_logprobs = list(old_token_logprobs[row])
            if len(row_logprobs) != len(completion_ids):
                raise ValueError(
                    "one old_logprobs list does not match its completion length"
                )
            if not all(math.isfinite(value) for value in row_logprobs):
                raise ValueError("old log-probabilities must be finite")
            old_logprobs[row, start:stop] = torch.tensor(
                row_logprobs, dtype=torch.float32
            )

    return CausalBatch(
        input_ids=input_ids,
        attention_mask=attention_mask,
        action_mask=action_mask,
        prompt_lengths=prompt_lengths,
        completion_lengths=completion_lengths,
        old_logprobs=old_logprobs,
    )


def rollout_batch(records: Sequence[dict[str, Any]], pad_token_id: int) -> "CausalBatch":
    return causal_batch(
        [record["prompt_token_ids"] for record in records],
        [record["completion_token_ids"] for record in records],
        pad_token_id,
        [record["old_logprobs"] for record in records],
    )


def ensure_padding_token(tokenizer: Any) -> None:
    if tokenizer.pad_token_id is None:
        if tokenizer.eos_token_id is None:
            raise ValueError("tokenizer has neither a pad token nor an EOS token")
        tokenizer.pad_token = tokenizer.eos_token

In [ ]:
toy = causal_batch([[11, 12, 13], [11, 12]], [[21, 22], [23]], pad_token_id=0)
for name in ("input_ids", "attention_mask", "action_mask"):
    print(name, getattr(toy, name).tolist())
assert toy.action_mask.sum(dim=1).tolist() == [2, 1]
assert toy.action_mask[0].tolist() == [False, False, True, True]
print("Response-predicted token IDs:", toy.input_ids[:, 1:][toy.action_mask].tolist())

## 5. OPSD objective and Jensen–Shannon divergence

Let $z_i$ be the verified solution and $\phi$ the frozen **initial** teacher parameters. The sampled response remains $y_i\sim\pi_{\theta_{\mathrm{old}}}(\cdot\mid x_i)$, even if incorrect. Define

$$p_{it}(v)=\pi_\theta(v\mid x_i,y_{i,<t}),\quad
q_{it}(v)=\operatorname{sg}[\pi_\phi(v\mid x_i,z_i,y_{i,<t})],\quad
u_{it}=\tfrac12(p_{it}+q_{it}).$$

The default objective is

$$\operatorname{JSD}(p,q)=\tfrac12\operatorname{KL}(p\|u)+\tfrac12\operatorname{KL}(q\|u),\qquad
\mathcal L_{\mathrm{OPSD}}=\frac1B\sum_i\frac1{T_i}\sum_t\operatorname{JSD}(p_{it},q_{it}).$$

Natural logarithms give $0\le\operatorname{JSD}\le\log 2$. JSD uses both distributions and their mixture. The teacher is detached, but the mixture depends on the student: the gradient must include that dependence in **both** KL terms. The code handles this with ordinary autograd. A supported alternative is `HP["divergence"] = "reverse_kl"`, which minimizes $\operatorname{KL}(p\|q)$.

The teacher does not generate a separate answer. Teacher forcing on the student's actual prefixes teaches how the informed model would proceed from mistakes as well as correct intermediate steps. Teacher and student completion slices therefore share token IDs but start at different absolute offsets. Tokenizer vocabulary mappings must match exactly.

The initial teacher stays fixed across all rounds; a second round must not silently replace it with the updated student. This recipe uses neither a value model nor PPO importance ratios. Stored scalar rewards are diagnostics, not weights in the distillation loss.

### Embedded model operations

The next cell defines architecture-aware loading, CUDA autocast, response-only vocabulary projections, and checkpoint saving. Model/tokenizer revisions are applied to Hub loads; local saved checkpoints load from their own files. Self-distillation detaches the teacher projection, transferring only its current logits chunk when the teacher is on CPU. No token-vocabulary truncation or approximation is introduced.

In [ ]:
def default_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def training_dtype(device: torch.device) -> torch.dtype:
    if device.type == "cuda" and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    if device.type == "cuda":
        return torch.float16
    return torch.float32


def load_tokenizer(model_name_or_path: str) -> Any:
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, use_fast=True, revision=MODEL_REVISION if model_name_or_path == MODEL_ID else None)
    ensure_padding_token(tokenizer)
    tokenizer.padding_side = "right"
    return tokenizer


def load_policy(
    model_name_or_path: str,
    device: torch.device,
    *,
    gradient_checkpointing: bool = False,
    attention_implementation: str = "sdpa",
    trainable: bool = True,
) -> nn.Module:
    config = AutoConfig.from_pretrained(model_name_or_path, revision=MODEL_REVISION if model_name_or_path == MODEL_ID else None)
    loader = (
        AutoModelForImageTextToText
        if config.model_type == "qwen3_5"
        else AutoModelForCausalLM
    )
    model = loader.from_pretrained(
        model_name_or_path,
        config=config,
        revision=MODEL_REVISION if model_name_or_path == MODEL_ID else None,
        # Adam needs FP32 master parameters at the small RL learning rates. BF16
        # parameter updates can otherwise round to zero. Autocast keeps compute cheap.
        dtype=torch.float32 if trainable else training_dtype(device),
        attn_implementation=attention_implementation,
    )
    model.config.use_cache = False
    if not trainable:
        model.requires_grad_(False)
    # Preserve the native checkpoint layout for vLLM; text training skips the vision tower.
    if config.model_type == "qwen3_5":
        model.config.text_config.use_cache = False
        model.model.visual.requires_grad_(False)
    if gradient_checkpointing:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
    return model.to(device)


def text_backbone(model: nn.Module) -> nn.Module:
    if model.config.model_type == "qwen3_5":
        return model.model.language_model
    return model.base_model


def batch_hidden_states(model: nn.Module, batch: CausalBatch) -> torch.Tensor:
    with compute_autocast(model):
        return text_backbone(model)(
            input_ids=batch.input_ids,
            attention_mask=batch.attention_mask,
            use_cache=False,
            return_dict=True,
        ).last_hidden_state


def compute_autocast(model: nn.Module):
    device = next(model.parameters()).device
    return torch.autocast(
        device_type=device.type,
        dtype=training_dtype(device),
        enabled=device.type == "cuda",
    )


def project_logits(model: nn.Module, states: torch.Tensor) -> torch.Tensor:
    with compute_autocast(model):
        return model.get_output_embeddings()(states)


def completion_hidden_states(
    model: nn.Module, batch: CausalBatch
) -> list[torch.Tensor]:
    hidden = batch_hidden_states(model, batch)
    return [
        hidden[i, p - 1 : p - 1 + c]
        for i, (p, c) in enumerate(zip(batch.prompt_lengths, batch.completion_lengths))
    ]


def hidden_logprobs(
    model: nn.Module, hidden: torch.Tensor, batch: CausalBatch, chunk_size: int = 32
) -> torch.Tensor:
    """Project only action states; recompute chunk softmaxes on backward."""
    if chunk_size < 1:
        raise ValueError("chunk_size must be positive")
    action_hidden = hidden[:, :-1][batch.action_mask]
    labels = batch.input_ids[:, 1:][batch.action_mask]

    def project(states: torch.Tensor, tokens: torch.Tensor) -> torch.Tensor:
        logits = project_logits(model, states).float()
        return -F.cross_entropy(logits, tokens, reduction="none")

    chunks = []
    for start in range(0, labels.numel(), chunk_size):
        states, tokens = (
            action_hidden[start : start + chunk_size],
            labels[start : start + chunk_size],
        )
        chunks.append(
            checkpoint(project, states, tokens, use_reentrant=False)
            if torch.is_grad_enabled()
            else project(states, tokens)
        )
    result = torch.zeros_like(batch.action_mask, dtype=torch.float32)
    return result.masked_scatter(batch.action_mask, torch.cat(chunks))


def policy_logprobs(
    model: nn.Module, batch: CausalBatch, chunk_size: int = 32
) -> torch.Tensor:
    return hidden_logprobs(model, batch_hidden_states(model, batch), batch, chunk_size)


def hidden_distillation_loss(
    student: nn.Module,
    teacher: nn.Module,
    student_rows: Sequence[torch.Tensor],
    teacher_rows: Sequence[torch.Tensor],
    divergence: str,
    chunk_size: int = 32,
) -> torch.Tensor:
    """Exact full-vocabulary divergence with bounded projection/softmax memory."""
    if chunk_size < 1 or not student_rows or len(student_rows) != len(teacher_rows):
        raise ValueError("invalid distillation chunk size or batch")
    if divergence not in ("jsd", "reverse_kl"):
        raise ValueError(f"unknown divergence: {divergence}")
    divergence_fn = jensen_shannon if divergence == "jsd" else reverse_kl

    def project(states: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # The live SDPO teacher head also needs stop-gradient, even when heads are tied.
        with torch.no_grad():
            teacher_logits = project_logits(teacher, targets).float().to(states.device)
        student_logits = project_logits(student, states).float()
        return divergence_fn(student_logits, teacher_logits).sum()

    losses = []
    for states, targets in zip(student_rows, teacher_rows):
        if states.shape[0] != targets.shape[0] or states.shape[0] == 0:
            raise ValueError("unaligned or empty distillation continuations")
        chunks = [
            checkpoint(
                project,
                states[i : i + chunk_size],
                targets[i : i + chunk_size],
                use_reentrant=False,
            )
            for i in range(0, states.shape[0], chunk_size)
        ]
        losses.append(torch.stack(chunks).sum() / states.shape[0])
    return torch.stack(losses).mean()


def require_matching_tokenizers(student_tokenizer: Any, teacher_name: str) -> None:
    teacher_tokenizer = load_tokenizer(teacher_name)
    if student_tokenizer.get_vocab() != teacher_tokenizer.get_vocab():
        raise ValueError(
            "teacher/reference and student must share the same token-to-ID vocabulary"
        )


def masked_mean(values: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    return (values * mask).sum() / mask.sum().clamp_min(1)


def reverse_kl(
    student_logits: torch.Tensor, teacher_logits: torch.Tensor
) -> torch.Tensor:
    """Per-token KL(student || stop-gradient teacher)."""
    student_logp = F.log_softmax(student_logits.float(), dim=-1)
    teacher_logp = F.log_softmax(teacher_logits.detach().float(), dim=-1)
    student_p = student_logp.exp()
    return (student_p * (student_logp - teacher_logp)).sum(dim=-1)


def jensen_shannon(
    student_logits: torch.Tensor, teacher_logits: torch.Tensor
) -> torch.Tensor:
    """Per-token symmetric JSD with equal mixture weights."""
    student_logp = F.log_softmax(student_logits.float(), dim=-1)
    teacher_logp = F.log_softmax(teacher_logits.detach().float(), dim=-1)
    log_mixture = torch.logaddexp(student_logp, teacher_logp) - 0.6931471805599453
    student_term = (student_logp.exp() * (student_logp - log_mixture)).sum(dim=-1)
    teacher_term = (teacher_logp.exp() * (teacher_logp - log_mixture)).sum(dim=-1)
    return 0.5 * (student_term + teacher_term)


def save_policy(model: nn.Module, tokenizer: Any, output_dir: str | Path) -> None:
    path = Path(output_dir)
    path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(path)
    tokenizer.save_pretrained(path)

In [ ]:
student = torch.tensor([[1.0, 0.0, -0.5]], requires_grad=True)
teacher = torch.tensor([[0.0, 1.0, -0.5]], requires_grad=True)
loss = jensen_shannon(student, teacher).mean()
loss.backward()
assert 0 <= loss.item() <= math.log(2)
assert teacher.grad is None
print(f"Jensen–Shannon divergence: {loss.item():.4f}")

## 6. OPSD's privileged reference

OPSD uses the shared rollout JSONL below. Every row must include a nonempty `reference`: a worked GSM8K solution or the normalized SVAMP equation-and-answer text. The reference is appended only to the teacher's private context. The ordinary student still sees `prompt` alone.

An illustrative fragment is:

```json
{"prompt":"What is 2 + 3?","answer":"5","reference":"Add the two quantities: 2 + 3 = 5. Final answer: 5","completion":"Final answer: 4","reward":0.0}
```

The complete record also needs exact prompt/completion token IDs and model metadata. A wrong completion is not automatically discarded: the teacher scores the student's visited prefixes. `answer` drives reward diagnostics, while `reference` provides the informed teacher context. If your custom data has only an answer, you can construct an answer-only reference, but that is a different supervision setting and should be reported as such.

### Shared on-policy rollout JSONL

For SDPO, OPSD, PPO, and GRPO, generation preserves the common dataset fields and adds:

| Field | Type / length | Purpose |
|---|---|---|
| `group_id` | string | Source question identity; same for all answers to that question |
| `prompt_token_ids` | list of $P$ integers | Exact rendered ordinary prompt supplied to generation |
| `completion` | string | Decoded answer for verifier/inspection |
| `completion_token_ids` | list of $T$ integers | Original generated actions, including sampled EOS if present |
| `old_logprobs` | list of $T$ finite floats | Log probability of each sampled token under the behavior policy |
| `reward` | finite float | Here 0 or 1 from the task's math, Python-test, or tool-call verifier |
| `feedback` | string | Verifier message, consumed by SDPO only |
| `model` | string | Checkpoint that generated this buffer; validated before training |
| `sampling` | object | `temperature` and `top_p` used for sampling |
| `enable_thinking` | Boolean | Must match the prompt rendering convention |
| `finish_reason` | string | `stop` or `length`; useful for truncation diagnostics |

For example, the **structural** fields might look like this (IDs/log probabilities are illustrative, not valid tokenizer output):

```json
{"group_id":"svamp:train:demo","prompt_token_ids":[11,12],"completion":"Final answer: 5","completion_token_ids":[21,22,23],"old_logprobs":[-0.7,-0.2,-1.1],"reward":1.0,"model":"<source checkpoint>","sampling":{"temperature":1.0,"top_p":1.0},"enable_thinking":false,"finish_reason":"stop"}
```

Never decode then re-encode a buffer to reconstruct actions; tokenization changes can invalidate likelihood ratios and prefix alignment. PPO and GRPO require **temperature 1, top-p 1, and top-k disabled** so stored likelihoods correspond to the unmodified policy distribution. No inherited repetition penalty or generation filter should silently change it. The shared generation code makes those choices explicitly and rejects incompatible policy-gradient buffers. Greedy evaluation is a separate protocol.

### Embedded teacher-context construction

These functions tokenize the private teacher prompt separately, then append the exact original completion IDs. Each completion slice is aligned to its own prompt length. OPSD uses a verified reference; SDPO uses feedback. The ordinary student and evaluator never receive these privileged contexts.

In [ ]:
OPSD_SYSTEM = (
    "You are a solution-aware teacher. Use the private verified solution to evaluate how a "
    "student should reason, but score the student's existing continuation token by token."
)


SDPO_SYSTEM = (
    "You are a retrospective teacher. Use the environment feedback to judge and improve the "
    "attempt as it unfolds. Score the existing continuation rather than writing a separate answer."
)


def opsd_teacher_prompt(record: dict[str, Any]) -> str:
    return (
        f"Problem:\n{record['prompt']}\n\n"
        f"Private verified solution:\n{record['reference']}\n\n"
        "Re-evaluate the student's attempted reasoning with this privileged information."
    )


def sdpo_teacher_prompt(record: dict[str, Any]) -> str:
    return (
        f"Problem:\n{record['prompt']}\n\n"
        f"Feedback received after the attempt:\n{record['feedback']}\n\n"
        "Re-evaluate the attempted reasoning in hindsight."
    )


def teacher_batch(
    records: Sequence[dict[str, Any]],
    tokenizer: Any,
    method: str,
    max_seq_length: int | None = None,
) -> CausalBatch:
    if method == "opsd":
        contexts = [
            chat_prompt_ids(tokenizer, opsd_teacher_prompt(row), OPSD_SYSTEM)
            for row in records
        ]
    elif method == "sdpo":
        contexts = [
            chat_prompt_ids(tokenizer, sdpo_teacher_prompt(row), SDPO_SYSTEM)
            for row in records
        ]
    else:
        raise ValueError(f"unknown teacher context method: {method}")
    completions = [row["completion_token_ids"] for row in records]
    if max_seq_length is not None and any(
        len(p) + len(c) > max_seq_length for p, c in zip(contexts, completions)
    ):
        raise ValueError(
            "privileged teacher context exceeds MAX_SEQ_LENGTH; increase the limit or shorten the data"
        )
    return causal_batch(contexts, completions, int(tokenizer.pad_token_id))

## 7. Model loading, memory, and optimization components

The code uses the model's actual architecture. Qwen3.5 is loaded through its multimodal auto class, trains its text backbone, freezes vision weights, and saves a native compatible checkpoint. LFM2.5 uses its causal-LM class. Both can use hybrid decoder operations, so simply replacing them with a generic Transformer block would change the model.

| Component | Implementation and reason |
|---|---|
| Trainable parameters | FP32 master weights so small full-parameter updates do not round away |
| Compute dtype | CUDA BF16 when supported, otherwise CUDA FP16 with gradient scaling; CPU FP32 |
| Attention | SDPA by default; optional eager or supported flash-attention paths |
| Checkpointing | Recompute intermediate activations during backward to trade compute for lower memory |
| Vocabulary head | Project only completion-predicting hidden states, in `CHUNK_SIZE` token chunks |
| Distribution losses | Exact vocabulary softmax/KL/JSD; chunking bounds temporary memory without truncating the vocabulary |
| Teacher/reference | Evaluation mode, no gradients or optimizer state; OPSD/GRPO keep an additional frozen model (OPSD on CPU by default) |
| Optimizer | FP32 AdamW, weight decay 0, global gradient-norm clipping at 1 |
| Accumulation | Average microbatch losses over `ACCUMULATION`; correctly rescale a shorter final window |
| Saved artifacts | Native policy config/weights/tokenizer; PPO additionally saves its scalar value head |

For microbatch loss $\mathcal L_k$, a full accumulation window applies

$$g=\frac1K\sum_{k=1}^{K}\nabla\mathcal L_k,\qquad g\leftarrow g\min\left(1,\frac1{\|g\|_2}\right),\qquad \theta\leftarrow\operatorname{AdamW}(\theta,g).$$

`K=ACCUMULATION` for a full window; a partial window uses its actual microbatch count. Unequal token counts can still make accumulation differ from one single global token-weighted batch because each microbatch has its own normalized loss. All optimization here is full-parameter training of the text policy, not adapters. Optimizer state is restarted at each round; weights and PPO's value head persist.

Start with the preset sequence limit, small batch size, checkpointing, and token chunks. A total limit must cover the ordinary prompt **and**, for self-distillation, the longer private teacher or rewriting prompt plus the response. Overlength data raises an error rather than silently dropping informative tokens. `MAX_TOKENS` controls generation length and therefore reward/truncation behavior; it is not just a memory setting.

### Embedded optimizer and buffer checks

The optimizer keeps FP32 master parameters, uses CUDA gradient scaling when BF16 is unavailable, clips the global gradient norm, and averages a partial accumulation window correctly. Buffer checks reject test records, wrong source checkpoints, overlength sequences, nonfinite rewards, and policy-gradient sampling mismatches. GRPO also validates complete same-question groups.

In [ ]:
def validate_training_args(args: SimpleNamespace) -> None:
    for name in (
        "gradient_accumulation_steps",
        "max_seq_length",
        "logit_chunk_size",
        "batch_size",
        "groups_per_batch",
        "epochs",
        "ppo_epochs",
        "policy_epochs",
    ):
        if hasattr(args, name) and getattr(args, name) < 1:
            raise ValueError(f"--{name.replace('_', '-')} must be positive")
    if (
        not math.isfinite(args.learning_rate)
        or not math.isfinite(args.max_grad_norm)
        or args.learning_rate <= 0
        or args.max_grad_norm <= 0
    ):
        raise ValueError("learning rate and max gradient norm must be positive")
    for name in ("clip_epsilon", "value_clip_epsilon"):
        if hasattr(args, name) and not 0 < getattr(args, name) < 1:
            raise ValueError(f"{name} must be in (0, 1)")
    for name in ("gamma", "gae_lambda"):
        if hasattr(args, name) and not 0 <= getattr(args, name) <= 1:
            raise ValueError(f"{name} must be in [0, 1]")
    if hasattr(args, "beta") and (not math.isfinite(args.beta) or args.beta < 0):
        raise ValueError("beta must be finite and nonnegative")


def validate_rollouts(
    records: list[dict[str, Any]],
    model: str,
    *,
    policy_gradient: bool = False,
    max_seq_length: int,
) -> None:
    if not records:
        raise ValueError("rollout file is empty")
    for row in records:
        if row.get("split") == "test":
            raise ValueError("training rollouts contain held-out test examples")
        if row.get("model") is not None and row["model"] != model:
            raise ValueError(
                "rollouts came from a different checkpoint; regenerate with MODEL"
            )
        if (
            len(row["prompt_token_ids"]) + len(row["completion_token_ids"])
            > max_seq_length
        ):
            raise ValueError(
                "rollout exceeds MAX_SEQ_LENGTH; regenerate with a shorter token budget"
            )
        if not math.isfinite(float(row["reward"])):
            raise ValueError("rollout reward must be finite")
        if policy_gradient:
            sampling = row.get("sampling", {})
            if (
                sampling.get("temperature", 1.0) != 1.0
                or sampling.get("top_p", 1.0) != 1.0
            ):
                raise ValueError(
                    "PPO/GRPO require untempered, untruncated rollouts (temperature=top-p=1)"
                )


class AccumulatingOptimizer:
    """Average microbatch gradients, including an incomplete final window."""

    def __init__(
        self,
        model: torch.nn.Module,
        learning_rate: float,
        steps: int,
        max_grad_norm: float,
    ):
        self.model = model
        self.steps = steps
        self.max_grad_norm = max_grad_norm
        self.pending = 0
        self.optimizer = torch.optim.AdamW(
            (p for p in model.parameters() if p.requires_grad),
            lr=learning_rate,
            weight_decay=0.0,
            fused=next(model.parameters()).device.type == "cuda",
        )
        self.optimizer.zero_grad(set_to_none=True)
        self.scaler = torch.amp.GradScaler(
            "cuda",
            enabled=(
                next(model.parameters()).device.type == "cuda"
                and not torch.cuda.is_bf16_supported()
            ),
        )

    def backward(self, loss: torch.Tensor) -> None:
        if not torch.isfinite(loss):
            raise FloatingPointError("non-finite training loss")
        self.scaler.scale(loss / self.steps).backward()
        self.pending += 1
        if self.pending == self.steps:
            self.flush()

    def flush(self) -> None:
        if not self.pending:
            return
        self.scaler.unscale_(self.optimizer)
        if self.pending != self.steps:
            for parameter in self.model.parameters():
                if parameter.grad is not None:
                    parameter.grad.mul_(self.steps / self.pending)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.max_grad_norm)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad(set_to_none=True)
        self.pending = 0

### Optional offline smoke fixture

This cell only acts when `STANDALONE_SMOKE=1`. It creates a random two-layer **real model architecture**, a tiny tokenizer, and a local native checkpoint. Together with the synthetic data in `prepare_data`, this makes end-to-end offline validation self-contained. A random tiny model is not evidence about the pretrained target model.

In [ ]:
def tiny_architecture():
    from transformers import (
        Qwen3_5Config,
        Qwen3_5ForConditionalGeneration,
        Qwen3_5TextConfig,
        Qwen3_5VisionConfig,
    )
    return Qwen3_5ForConditionalGeneration(
        Qwen3_5Config(
            text_config=Qwen3_5TextConfig(
                vocab_size=32,
                hidden_size=16,
                intermediate_size=32,
                num_hidden_layers=2,
                num_attention_heads=2,
                num_key_value_heads=1,
                head_dim=8,
                layer_types=["linear_attention", "full_attention"],
                linear_key_head_dim=8,
                linear_value_head_dim=8,
                linear_num_key_heads=2,
                linear_num_value_heads=2,
                rope_parameters={
                    "rope_type": "default",
                    "rope_theta": 10000.0,
                    "partial_rotary_factor": 1.0,
                    "mrope_section": [1, 1, 2],
                },
                eos_token_id=2,
                pad_token_id=0,
            ),
            vision_config=Qwen3_5VisionConfig(
                depth=1,
                hidden_size=16,
                intermediate_size=32,
                num_heads=2,
                out_hidden_size=16,
                num_position_embeddings=16,
            ),
            image_token_id=29,
            video_token_id=30,
        )
    )

In [ ]:
def create_tiny_seed():
    from tokenizers import Tokenizer
    from tokenizers.models import WordLevel
    from tokenizers.pre_tokenizers import Whitespace
    from transformers import PreTrainedTokenizerFast
    words = ["[PAD]", "[UNK]", "[EOS]", "System", "User", "Assistant", ":", "Final", "answer", "+", "=", "?", ".", "1", "2", "3", "4", "5", "6", "7", "8"]
    tokenizer_backend = Tokenizer(WordLevel({w: i for i, w in enumerate(words)}, unk_token="[UNK]"))
    tokenizer_backend.pre_tokenizer = Whitespace()
    tokenizer = PreTrainedTokenizerFast(tokenizer_object=tokenizer_backend, pad_token="[PAD]",
                                       unk_token="[UNK]", eos_token="[EOS]")

    model = tiny_architecture()
    model.generation_config.eos_token_id = 2
    model.generation_config.pad_token_id = 0
    path = RUN_DIR / "tiny_seed"
    model.save_pretrained(path)
    tokenizer.save_pretrained(path)
    return str(path)

if SMOKE:
    MODEL = create_tiny_seed()
    clear_memory()
    print("Created tiny random smoke model:", MODEL)

### How the selected datasets enter OPSD training

The current policy generates a response from the public task. The initial frozen teacher receives the worked solution, verified reference program, or reference call array in its private context. `train_once` aligns both models on the same generated continuation and minimizes the configured full-vocabulary divergence. The teacher stays at the initial checkpoint across rounds. Private references are never supplied to the evaluated policy, and scalar rewards do not weight the distillation objective.

The executable gate below checks the selected normalized tasks before training and saves `dataset_recipe.json`. The table shows **the rows actually sampled**, so a selected source with zero sampled rows is not presented as contributing to the update. Adjust `TRAIN_SOURCES`, `DOMAINS`, and per-domain `TRAIN_LIMITS` in the configuration to change the mixture. Use both `gsm8k` and `svamp` in the math source list to combine their fitting pools; add optional authenticated `xlam` alongside `hermes` for both tool sources. HumanEval and BFCL remain final benchmarks.

Generated buffers are checked again inside `train_once` before allocating the training model. Their public prompt, domain, and verifier must match one of the selected training tasks, and they must contain the objective's required fields. This rejects validation/test leakage and accidental loss of domain metadata during data infusion. All domains still share the requested model, tokenizer, response mask, and recipe loss. Single-turn tool-call checking supplies a verifiable action target; full multi-turn agent datasets require a different environment/trajectory loop and are excluded here.


In [ ]:
RECIPE_SIGNAL = 'Public-task rollout + private verified reference → frozen teacher → full-vocabulary distillation'

def check_recipe_examples(examples):
    """Check that fitting data supplies this recipe's actual learning signal."""
    if not examples:
        raise ValueError("No fitting examples were selected")
    if ALGORITHM == "grpo" and SAMPLES < 2:
        raise ValueError("GRPO requires at least two responses per fitting task")
    kinds = {"math": "numeric", "coding": "python_tests", "tool_use": "tool_calls"}
    counts = Counter()
    for row in examples:
        if row["split"] != "train" or row["dataset_name"] in ("humaneval", "bfcl_simple"):
            raise ValueError("Evaluation-only data cannot enter the fitting recipe")
        if row["domain"] not in SELECTED_DOMAINS or row["verifier"].get("kind") != kinds[row["domain"]]:
            raise ValueError("The fitting task lacks its domain's supported verifier")
        if not row.get("prompt", "").strip():
            raise ValueError("A fitting task requires a public prompt")
        if ALGORITHM in ("sdft", "opsd") and (
            not row.get("reference", "").strip() or verify_completion(row["reference"], row)["reward"] != 1
        ):
            raise ValueError(f"{ALGORITHM.upper()} requires a correct reference: {row['id']}")
        counts[(row["domain"], row["dataset_name"])] += 1
    report = {"algorithm": ALGORITHM, "learning_signal": RECIPE_SIGNAL,
              "sources": [{"domain": d, "dataset": name, "training_tasks": n,
                           "verifier": kinds[d]} for (d, name), n in sorted(counts.items())],
              "verified_references_required": ALGORITHM in ("sdft", "opsd"),
              "responses_per_task": SAMPLES, "evaluation_sources_excluded": ["humaneval", "bfcl_simple"]}
    (RUN_DIR / "dataset_recipe.json").write_text(json.dumps(report, indent=2) + "\n")
    lines = ["| Domain | Training source | Tasks | Verifier |", "|---|---|---:|---|"]
    lines.extend(f"| {d} | {name} | {n} | {kinds[d]} |" for (d, name), n in sorted(counts.items()))
    display(Markdown("\n".join(lines)))
    print("Actual objective data flow:", RECIPE_SIGNAL)
    return report


def check_recipe_buffer(records):
    """Keep generated fitting buffers within their selected tasks and objective schema."""
    required = {"sdft": ("selected_response", "rewrite_verified"), "sdpo": ("feedback",),
                "opsd": ("reference",), "ppo": ("reward", "old_logprobs", "completion_token_ids"),
                "grpo": ("reward", "old_logprobs", "completion_token_ids", "group_id")}[ALGORITHM]
    selected = {row["id"]: row for row in train_data}
    if not records:
        raise ValueError("The recipe generated an empty fitting buffer")
    for row in records:
        source_id = row.get("group_id", row["id"])
        if row.get("split") != "train" or source_id not in selected:
            raise ValueError("A fitting buffer contains an unselected or held-out task")
        original = selected[source_id]
        if any(row.get(key) != original[key] for key in ("prompt", "domain", "verifier")):
            raise ValueError("A fitting buffer changed its public task or domain verifier")
        if any(key not in row or row[key] is None for key in required):
            raise ValueError(f"{ALGORITHM.upper()} buffer is missing objective fields: {required}")
        for key in ("selected_response", "feedback", "reference"):
            if key in required and not str(row[key]).strip():
                raise ValueError(f"The recipe requires nonempty {key}")
    if ALGORITHM == "grpo":
        groups = Counter(row["group_id"] for row in records)
        if SAMPLES < 2 or any(count != SAMPLES for count in groups.values()):
            raise ValueError("GRPO requires complete same-task response groups")


In [ ]:
train_data, validation_data, test_data, paths = prepare_data()
print({name: len(read_jsonl(path)) for name, path in paths.items()})
for domain in SELECTED_DOMAINS:
    print(domain, "training row:", json.dumps(next(r for r in train_data if r["domain"] == domain), indent=2))
print("Source/filter/split details:", RUN_DIR / "data_protocol.json")
protocol = json.loads((RUN_DIR / "data_protocol.json").read_text())
print("Domain selection/filter counts:", json.dumps(protocol["filtering"], indent=2))
print("Source normalization exclusions:", {name: details.get("normalization_rejections", {})
    for name, details in protocol["sources"].items() if details.get("normalization_rejections")})
recipe_data_report = check_recipe_examples(train_data)

### Embedded generation

Generation is serial to bound GPU memory. A fresh `GenerationConfig` disables inherited filters and penalties. Training rollouts sample the unmodified policy at temperature/top-p 1 with top-k disabled; evaluation and SDFT rewriting are greedy. The chosen-token likelihoods are read from the original generation scores, and sampled EOS actions are preserved. Models are function-local and released between stages.

In [ ]:
def generate_with_transformers(
    model_name_or_path: str,
    prompt_ids: Sequence[Sequence[int]],
    *,
    samples_per_prompt: int,
    max_tokens: int,
    temperature: float,
    top_p: float,
    seed: int,
    max_model_len: int = 2048,
    require_logprobs: bool = True,
    **_: Any,
) -> list[Any]:
    """Portable fallback; generate serially to keep memory bounded on CPU/small GPUs."""
    from types import SimpleNamespace

    import torch


    if any(len(ids) + max_tokens > max_model_len for ids in prompt_ids):
        raise ValueError("prompt + generation budget exceeds MAX_SEQ_LENGTH")
    torch.manual_seed(seed)
    device = default_device()
    model = load_policy(model_name_or_path, device, trainable=False).eval()
    tokenizer = load_tokenizer(model_name_or_path)
    generation = GenerationConfig(
        max_new_tokens=max_tokens,
        do_sample=temperature > 0,
        temperature=temperature if temperature > 0 else 1.0,
        top_p=top_p,
        top_k=0 if temperature > 0 else None,
        repetition_penalty=1.0,
        eos_token_id=model.generation_config.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        bos_token_id=tokenizer.bos_token_id,
        return_dict_in_generate=True,
        output_scores=require_logprobs,
        use_cache=True,
    )
    outputs = []
    with torch.inference_mode():
        for ids in prompt_ids:
            samples = []
            for _ in range(samples_per_prompt):
                inputs = torch.tensor([ids], device=device)
                result = model.generate(
                    input_ids=inputs,
                    attention_mask=torch.ones_like(inputs),
                    generation_config=generation,
                )
                tokens = result.sequences[0, len(ids) :].tolist()
                logprobs = None
                if require_logprobs:
                    transitions = model.compute_transition_scores(
                        result.sequences, result.scores, normalize_logits=True
                    )[0].tolist()
                    logprobs = [
                        {token: SimpleNamespace(logprob=value)}
                        for token, value in zip(tokens, transitions)
                    ]
                eos = generation.eos_token_id
                eos_ids = eos if isinstance(eos, list) else [eos]
                samples.append(
                    SimpleNamespace(
                        text=tokenizer.decode(tokens, skip_special_tokens=True),
                        token_ids=tokens,
                        logprobs=logprobs,
                        finish_reason="stop"
                        if tokens and tokens[-1] in eos_ids
                        else "length",
                    )
                )
            outputs.append(SimpleNamespace(prompt_token_ids=list(ids), outputs=samples))
    return outputs

In [ ]:
def chosen_logprobs(
    token_ids: Sequence[int], logprob_rows: Sequence[dict[int, Any]]
) -> list[float]:
    values: list[float] = []
    if len(token_ids) != len(logprob_rows):
        raise ValueError("vLLM returned different token and log-probability lengths")
    for token_id, candidates in zip(token_ids, logprob_rows):
        candidate = candidates.get(token_id)
        if candidate is None:
            # Some vLLM versions expose integer-like keys rather than plain int keys.
            candidate = next(
                (value for key, value in candidates.items() if int(key) == token_id),
                None,
            )
        if candidate is None:
            raise KeyError(f"sampled token {token_id} missing from vLLM logprobs")
        values.append(float(candidate.logprob))
    return values

def make_rollouts(model_name, examples, seed):
    tokenizer = load_tokenizer(model_name)
    prompt_ids = [chat_prompt_ids(tokenizer, row["prompt"]) for row in examples]
    outputs = generate_with_transformers(model_name, prompt_ids, samples_per_prompt=SAMPLES,
        max_tokens=MAX_TOKENS, temperature=1.0, top_p=1.0, seed=seed, max_model_len=MAX_SEQ_LENGTH)
    if len(outputs) != len(examples):
        raise ValueError("generation returned an incomplete prompt batch")
    records, rejected = [], 0
    for example, ids, output in zip(examples, prompt_ids, outputs):
        group = []
        for index, sample in enumerate(output.outputs):
            if not sample.token_ids:
                continue
            result = verify_completion(sample.text, example)
            row = {**example, "id": f"{example['id']}:{index}", "group_id": example["id"],
                   "prompt_token_ids": list(output.prompt_token_ids or ids), "completion": sample.text,
                   "completion_token_ids": list(sample.token_ids),
                   "old_logprobs": chosen_logprobs(sample.token_ids, sample.logprobs),
                   "reward": result["reward"], "feedback": result["feedback"][:MAX_FEEDBACK_CHARS],
                   "model": str(model_name), "sampling": {"temperature": 1.0, "top_p": 1.0},
                   "enable_thinking": False, "finish_reason": sample.finish_reason}
            group.append(row)
        if len(group) == SAMPLES:
            records.extend(group)
        else:
            rejected += 1  # A GRPO group is kept or removed as a whole.
    if not records:
        raise ValueError("No complete rollout groups remain; increase the context budget or shorten tasks")
    for domain in SELECTED_DOMAINS:
        selected = [r for r in records if r["domain"] == domain]
        if not selected:
            raise ValueError(f"All {domain} rollout groups were rejected; adjust the context budget")
        groups = {}
        for row in selected:
            groups.setdefault(row["group_id"], []).append(row["reward"])
        varied = sum(len(set(rewards)) > 1 for rewards in groups.values())
        print(f"{domain}: {len(selected)} rollouts, mean reward={sum(r['reward'] for r in selected) / len(selected):.3f}"
              + (f", groups with useful reward variance={varied}/{len(groups)}" if ALGORITHM == "grpo" else ""))
    print("Rejected incomplete/oversized rollout groups:", rejected)
    return records

## 8. The complete training function

The next cell contains the entire method-specific optimization loop. Every call it makes resolves to a function defined above, or to an installed external library. `HP` controls its actual hyperparameters. The initial `MODEL` remains the frozen OPSD teacher/GRPO reference across rounds, while `model_name` is the current student checkpoint. PPO snapshots all old critic values and targets before any update and reloads its saved value head in later rounds. SDFT has two supervised epochs by default; other methods have one optimization epoch.

No subprocess, local training script, or hidden framework trainer is invoked. Losses are printed per minibatch. Fresh training buffers are generated each round, and the optimizer starts anew for that round.

In [ ]:
def train_once(model_name, records, output, seed):
    args = SimpleNamespace(model=model_name, output=str(output), seed=seed,
                           teacher_model=MODEL, reference_model=MODEL, **HP)
    validate_training_args(args)
    check_recipe_buffer(records)

    validate_rollouts(records, args.model, policy_gradient=False, max_seq_length=args.max_seq_length)

    for row in records:
        if not row.get('reference'):
            raise ValueError("OPSD requires a non-empty 'reference' solution in every rollout")

    random.seed(args.seed)

    torch.manual_seed(args.seed)

    device = default_device()

    tokenizer = load_tokenizer(args.model)

    student = load_policy(args.model, device, gradient_checkpointing=args.gradient_checkpointing, attention_implementation=args.attention_implementation)

    require_matching_tokenizers(tokenizer, args.teacher_model or args.model)

    teacher = load_policy(args.teacher_model or args.model, REFERENCE_DEVICE, attention_implementation=args.attention_implementation, trainable=False)

    teacher.eval()

    teacher.requires_grad_(False)

    optimizer = AccumulatingOptimizer(student, args.learning_rate, args.gradient_accumulation_steps, args.max_grad_norm)

    step = 0

    for epoch in range(args.epochs):
        for rows in shuffled_batches(records, args.batch_size, args.seed + epoch):
            student_inputs = rollout_batch(rows, int(tokenizer.pad_token_id)).to(device)
            teacher_inputs = teacher_batch(rows, tokenizer, 'opsd', args.max_seq_length).to(REFERENCE_DEVICE)
            with torch.no_grad():
                teacher_rows = completion_hidden_states(teacher, teacher_inputs)
            student.train()
            student_rows = completion_hidden_states(student, student_inputs)
            loss = hidden_distillation_loss(student, teacher, student_rows, teacher_rows, args.divergence, args.logit_chunk_size)
            optimizer.backward(loss)
            step += 1
            print(f'epoch={epoch + 1} step={step} opsd_{args.divergence}={loss.item():.5f}')
        optimizer.flush()

    optimizer.flush()

    save_policy(student, tokenizer, args.output)

    print(f'saved OPSD policy to {args.output}')

### Embedded evaluation and benchmark reporting

Evaluation renders only `row["prompt"]`. It never supplies references, feedback, or expected answers to generation. Expected answers enter the verifier only after the completion is produced. These functions write both aggregate metrics and per-question predictions. Paired seed-versus-trained comparisons use identical question IDs.

In [ ]:
def score_outputs(examples, outputs):
    if not examples or len(examples) != len(outputs):
        raise ValueError("evaluation requires one output group per question")
    predictions = []
    for example, output in zip(examples, outputs):
        if len(output.outputs) != 1:
            raise ValueError("the benchmark protocol uses exactly one greedy response")
        sample = output.outputs[0]
        result = verify_completion(sample.text, example)
        predictions.append({"id": example["id"], "domain": example["domain"],
            "dataset_name": example["dataset_name"], "prompt": example["prompt"], "answer": example["answer"],
            "completion": sample.text, "predicted_answer": result["predicted"], "reward": result["reward"],
            "diagnostic": result["feedback"], "json_valid": result.get("json_valid"),
            "schema_valid": result.get("schema_valid"), "completion_tokens": len(sample.token_ids),
            "finish_reason": sample.finish_reason, "sample_index": 0})
    def summarize(rows):
        result = {"num_examples": len(rows), "accuracy": sum(r["reward"] for r in rows) / len(rows),
                  "mean_completion_tokens": sum(r["completion_tokens"] for r in rows) / len(rows),
                  "truncation_rate": sum(r["finish_reason"] == "length" for r in rows) / len(rows)}
        if all(r["domain"] == "tool_use" for r in rows):
            result.update(json_valid_rate=sum(r["json_valid"] for r in rows) / len(rows),
                          schema_valid_rate=sum(r["schema_valid"] for r in rows) / len(rows))
        return result
    by_domain = {d: summarize([r for r in predictions if r["domain"] == d])
                 for d in dict.fromkeys(r["domain"] for r in predictions)}
    metrics = summarize(predictions)
    # Mixed evaluations have domain-specific format metrics, never a mixed JSON-valid rate.
    metrics.pop("json_valid_rate", None)
    metrics.pop("schema_valid_rate", None)
    metrics.update(num_completions=len(predictions), by_domain=by_domain,
                   macro_domain_score=sum(r["accuracy"] for r in by_domain.values()) / len(by_domain),
                   protocol="one greedy full response; domain verifier; no teacher information")
    return metrics, predictions

def evaluate_model(model_name, examples, output):
    tokenizer = load_tokenizer(str(model_name))
    ids = [chat_prompt_ids(tokenizer, row["prompt"]) for row in examples]
    started = time.perf_counter()
    outputs = generate_with_transformers(str(model_name), ids, samples_per_prompt=1,
        max_tokens=MAX_TOKENS, temperature=0.0, top_p=1.0, seed=SEED,
        max_model_len=MAX_SEQ_LENGTH, require_logprobs=False)
    metrics, predictions = score_outputs(examples, outputs)
    metrics.update(model=str(model_name), seed=SEED, max_tokens=MAX_TOKENS,
                   max_seq_length=MAX_SEQ_LENGTH, samples_per_prompt=1,
                   enable_thinking=False, backend="transformers",
                   elapsed_seconds=time.perf_counter() - started, smoke=SMOKE)
    output = Path(output)
    output.write_text(json.dumps(metrics, indent=2) + "\n")
    write_jsonl(output.with_suffix(".predictions.jsonl"), predictions)
    clear_memory()
    return metrics

In [ ]:
def wilson_interval(accuracy, count, z=1.96):
    """Approximate 95% binomial interval for one greedy answer per question."""
    center = (accuracy + z * z / (2 * count)) / (1 + z * z / count)
    radius = z * (
        (accuracy * (1 - accuracy) / count + z * z / (4 * count * count)) ** 0.5
    )
    radius /= 1 + z * z / count
    return max(0, center - radius), min(1, center + radius)


def report_benchmarks(results, run_dir):
    lines = ["| Domain | Dataset / metric | Before | After | Change | N | After 95% interval | Scope |",
             "|---|---|---:|---:|---:|---:|---|---|"]
    paired, domain_rows = {}, []
    for dataset in dict.fromkeys(key[0] for key in results):
        baseline, trained = results[(dataset, "seed")], results[(dataset, "trained")]
        domain = next(iter(baseline["by_domain"]))
        predictions = {label: {r["id"]: r["reward"] for r in
            read_jsonl(run_dir / f"benchmark_{dataset}_{label}.predictions.jsonl")} for label in ("seed", "trained")}
        before, after = predictions["seed"], predictions["trained"]
        if before.keys() != after.keys() or len(before) != baseline["num_examples"]:
            raise ValueError("paired benchmarks require identical, unique question IDs")
        gained = sum(before[k] == 0 and after[k] == 1 for k in before)
        lost = sum(before[k] == 1 and after[k] == 0 for k in before)
        delta = (gained - lost) / len(before)
        lo, hi = wilson_interval(trained["accuracy"], len(after))
        metric = "BFCL simple JSON-argument match (adapted)" if dataset == "bfcl_simple" else METRIC_NAMES[domain]
        lines.append(f"| {domain} | {dataset}: {metric} | {baseline['accuracy']:.1%} | {trained['accuracy']:.1%} | "
                     f"{delta:+.1%} | {len(before)} | {lo:.1%}–{hi:.1%} | {trained['scope']} |")
        paired[dataset] = {"domain": domain, "metric": metric, "n": len(before), "gained": gained,
                          "lost": lost, "before": baseline["accuracy"], "after": trained["accuracy"],
                          "score_delta": delta, "before_wilson": wilson_interval(baseline["accuracy"], len(before)),
                          "after_wilson": (lo, hi)}
        domain_rows.append({"domain": domain, "dataset": dataset, **paired[dataset]})
        print(f"{dataset}: {gained} gained / {lost} lost; tokens {baseline['mean_completion_tokens']:.1f} → "
              f"{trained['mean_completion_tokens']:.1f}; truncation {baseline['truncation_rate']:.1%} → {trained['truncation_rate']:.1%}")
        if domain == "tool_use":
            for label, metrics in (("before", baseline), ("after", trained)):
                details = metrics["by_domain"][domain]
                print(f"  {label}: JSON valid={details['json_valid_rate']:.1%}, schema valid={details['schema_valid_rate']:.1%}")
    display(Markdown("\n".join(lines)))
    (run_dir / "paired_comparison.json").write_text(json.dumps(paired, indent=2) + "\n")
    (run_dir / "domain_comparison.json").write_text(json.dumps(domain_rows, indent=2) + "\n")
    fig, ax = plt.subplots(figsize=(max(6, len(paired) * 1.4), 3))
    names = list(paired)
    positions = list(range(len(names)))
    ax.bar([x - .18 for x in positions], [paired[n]["before"] for n in names], width=.36, label="Before")
    ax.bar([x + .18 for x in positions], [paired[n]["after"] for n in names], width=.36, label="After")
    ax.set(xticks=positions, xticklabels=names, ylim=(0, 1.05), ylabel="Domain-specific score", title="Same questions, decoding, and verifier")
    ax.tick_params(axis="x", rotation=25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(run_dir / "benchmark_before_after.png", dpi=150)
    plt.show()
    plt.close(fig)
    return paired

### Run generation, training, and validation

The seed is evaluated on the reserved validation partition first. Each round generates a buffer from the current checkpoint, saves that JSONL, calls the fully embedded trainer, and evaluates the resulting checkpoint on the same validation questions. Model references remain fixed to the seed. The predetermined last-round checkpoint is used for final testing; selecting by validation instead is possible, but must be recorded before looking at test scores.

In [ ]:
LOOP_DIR = RUN_DIR / "loop"
LOOP_DIR.mkdir(exist_ok=True)
validation_metrics = [evaluate_model(MODEL, validation_data, LOOP_DIR / "baseline_eval.json")]
current_model = MODEL
training_started = time.perf_counter()
for round_index in range(1, ROUNDS + 1):
    round_seed = SEED + round_index - 1
    buffer = make_rollouts(str(current_model), train_data, round_seed)
    buffer_path = LOOP_DIR / f"round_{round_index:02d}_rollouts.jsonl"
    write_jsonl(buffer_path, buffer)
    clear_memory()
    checkpoint_dir = LOOP_DIR / f"round_{round_index:02d}_model"
    train_once(str(current_model), buffer, checkpoint_dir, round_seed)
    clear_memory()
    current_model = checkpoint_dir
    metrics = evaluate_model(str(current_model), validation_data,
                             LOOP_DIR / f"round_{round_index:02d}_eval.json")
    validation_metrics.append(metrics)
    print(f"Round {round_index}: macro-domain validation score={metrics['macro_domain_score']:.1%}")
FINAL_MODEL = Path(current_model)
print(f"Training-stage wall time: {time.perf_counter() - training_started:.1f}s; saved {FINAL_MODEL}")

## 9. Inspect the generated buffer and validation curve

Before trusting the score, inspect what actually entered the trainer. The next cell inspects the final round's training records and performs a method-specific check. The validation table and plot then compare the seed with every trained round using identical held-out questions and greedy decoding.

A tiny validation partition is useful for checking the pipeline but gives a noisy estimate. Increase it before model selection, and repeat independent training seeds when estimating algorithm variability. Per-domain validation scores should guide changes to learning rate, epochs, response budget, or data—not the official test outputs in the next section.

In [ ]:
suffix = "sdft" if ALGORITHM == "sdft" else "rollouts"
buffer_path = LOOP_DIR / f"round_{ROUNDS:02d}_{suffix}.jsonl"
buffer = read_jsonl(buffer_path)
print("Generated training rows:", len(buffer), "| Fields:", sorted(buffer[0]))

print("Student prompt:", buffer[0]["prompt"])
print("Verified reference:", buffer[0]["reference"])
print("Teacher prompt:\n", opsd_teacher_prompt(buffer[0]))

Inspect finite JSD and validation accuracy together. A small divergence can mean the two contexts already produce similar predictions, not necessarily that answers are correct. Check reference quality, tokenizer matching, completion alignment, and how often outputs hit the response budget.

To change the update configuration, edit `HP` before training and inspect the embedded `train_once` function. Changing epochs reuses a buffer; changing rounds generates fresh responses. Regenerate buffers when switching source checkpoints rather than editing their provenance metadata.

In [ ]:
labels = ["baseline"] + [f"round_{i:02d}" for i in range(1, ROUNDS + 1)]
validation_metrics = [json.loads((LOOP_DIR / f"{label}_eval.json").read_text()) for label in labels]
lines = ["| Checkpoint | Domain | Score | Truncated |", "|---|---|---:|---:|"]
fig, ax = plt.subplots(figsize=(6, 3))
for domain in SELECTED_DOMAINS:
    scores = []
    for label, row in zip(labels, validation_metrics):
        details = row["by_domain"][domain]
        scores.append(details["accuracy"])
        lines.append(f"| {label} | {domain} | {details['accuracy']:.1%} | {details['truncation_rate']:.1%} |")
    ax.plot(labels, scores, marker="o", label=domain)
ax.plot(labels, [r["macro_domain_score"] for r in validation_metrics], linestyle="--", label="macro domains")
display(Markdown("\n".join(lines)))
ax.set(ylim=(-0.02, 1.02), ylabel="Domain-specific validation score", title=f"{ALGORITHM.upper()} — validation")
ax.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / "validation_curve.png", dpi=150)
plt.show()
plt.close(fig)


## 10. Before/after final benchmarks, by domain

The untrained baseline is the **original requested pretrained checkpoint**, before any notebook updates; it is not a randomly initialized model. The trained checkpoint is the final policy after `ROUNDS`. Each final benchmark compares these two checkpoints using identical public prompts, shared context/generation budgets, one greedy response, thinking disabled, and the same domain verifier. No reference body, expected calls, hidden tests, or verifier feedback is passed to either evaluated policy. Official source-provided function docstrings/interfaces remain public.

Math evaluates GSM8K and SVAMP official tests. Coding evaluates MBPP official `test` and HumanEval official `test` with the instruction-style constrained worker described above. Tool use evaluates a source-distribution holdout plus the independent BFCL simple-call dataset with the **adapted** JSON argument matcher. Whole agent execution, multi-turn success, and the official BFCL leaderboard aggregate are outside this protocol. Inapplicable datasets are not mixed into a recipe just because they contain conversations.

`BENCHMARK_LIMIT=32` selects a seeded deterministic subset per prepared benchmark; `None` uses every prepared row. The report shows subset/full-prepared scope and the number excluded by context filtering. Toy smoke fixtures are always labeled separately. Saving original task IDs and manifest hashes makes before/after pairing auditable. Never tune on these final tests.

For binary outcomes $R_i$, $\widehat s_d=N_d^{-1}\sum_iR_i$. Coding's score is greedy single-sample execution pass@1; tool score is structured call accuracy; math score is numeric exact match. Report these domain metrics separately instead of advertising a combined number as mathematical or agent accuracy. Validation includes an equally weighted macro-domain score $D^{-1}\sum_ds_d$ to track a mixture, alongside each domain's score. Final tests remain separate by dataset and domain.

The before/after table includes $\Delta s=(\mathrm{gained}-\mathrm{lost})/N$ on identical IDs, the number of questions, and a 95% Wilson binomial interval. Per-question predictions preserve failures, diagnostics, token counts, and truncation; tool JSON/schema validity is also printed. Negative gains are meaningful results. Wilson intervals describe question-sample uncertainty under simplifying assumptions, not training-seed variation or proof of causal algorithm superiority. More samples, independent training seeds, and the publisher's full harness are needed for a leaderboard claim.

`paired_comparison.json`, `domain_comparison.json`, and `benchmark_before_after.png` capture actual measured scores; they never contain fabricated improvements. Context/normalization restrictions and the constrained coding execution protocol must accompany any exported results.

In [ ]:
benchmark_results = {}
protocol = json.loads((RUN_DIR / "data_protocol.json").read_text())
for dataset, available in test_data.items():
    examples = available if BENCHMARK_LIMIT is None else available[:BENCHMARK_LIMIT]
    prepared = protocol["sources"][f"{dataset}/prepared_test"]
    print(dataset, "available:", len(available), "context excluded:", prepared["context_excluded"])
    for label, model_name in {"seed": MODEL, "trained": str(FINAL_MODEL)}.items():
        output = RUN_DIR / f"benchmark_{dataset}_{label}.json"
        metrics = evaluate_model(model_name, examples, output)
        scope = "SMOKE CHECK — toy data, not benchmark" if SMOKE else (
            "full prepared test" if len(examples) == len(available) else "prepared test subset")
        metrics.update(dataset=dataset, label=label, scope=scope, coverage=prepared,
                       metric="BFCL simple JSON-argument match (adapted)" if dataset == "bfcl_simple"
                       else METRIC_NAMES[examples[0]["domain"]])
        output.write_text(json.dumps(metrics, indent=2) + "\n")
        benchmark_results[(dataset, label)] = metrics
paired_results = report_benchmarks(benchmark_results, RUN_DIR)


## 11. Inspect errors, preserve artifacts, and interpret results

The next cell reads the **saved** prediction records from the main benchmark. Each record contains question ID, sample index, prompt, expected answer, generated text, extracted answer, reward, token count, and finish reason. Reading the expected answer after evaluation is an analysis step; it did not enter the evaluation model context. Inspect some successes too: a correct final number does not guarantee a valid explanation.

Record your chosen hyperparameters and data/model revisions before running the final test. Test-error inspection is for reporting and planning a subsequent experiment; after making changes based on it, use a fresh assessment set if you want an unbiased final claim. Across algorithms, these presets use different model sizes and training distributions, so raw cross-notebook accuracy is not an isolated causal comparison of optimization methods.

Useful artifacts are `config.json` (settings, revisions, package versions, device), `data_protocol.json` (partition IDs, hashes, source counts), `loop/round_XX_model/` (native checkpoint), generated JSONL buffers, per-round validation metrics/predictions, `benchmark_<dataset>_<seed|trained>.json`, their `.predictions.jsonl` companions, `paired_comparison.json`, `domain_comparison.json`, `validation_curve.png`, and `benchmark_before_after.png`. Printed training diagnostics remain in the notebook outputs. Wall time includes model loading and is not a pure token-throughput benchmark.

If CUDA runs out of memory, reduce the microbatch/group size where the trainer supports it, lower concurrency, and use smaller projection chunks while retaining checkpointing. Reducing sequence/response budgets changes the task and must be reported. If rollout provenance or group validation fails, regenerate from the intended checkpoint instead of editing metadata. If answer extraction fails, inspect explicit final-answer formatting and unfinished thinking. The notebooks make no guarantee that a short run improves the target models.

For mixed runs, compare each domain and benchmark separately. The error cell prints one failure per benchmark with its verifier diagnostic, so coding syntax/timeouts and tool JSON/schema errors are visible alongside math errors. Keep context-exclusion counts and adapted benchmark labels when sharing results.

In [ ]:
for dataset in test_data:
    predictions = read_jsonl(RUN_DIR / f"benchmark_{dataset}_trained.predictions.jsonl")
    errors = [r for r in predictions if r["reward"] == 0]
    print(dataset, f"{len(errors)}/{len(predictions)} failures")
    for row in errors[:1] or predictions[:1]:
        print(json.dumps({k: row[k] for k in ("id", "domain", "answer", "completion", "diagnostic", "finish_reason")}, indent=2))


### Optional Colab export

Set `DOWNLOAD_RESULTS=True` after the experiment if you want Colab to download a ZIP of metrics, JSONL records, and plots. Checkpoints can be large: set `INCLUDE_CHECKPOINTS=True` only when you want their weights included. On a local Jupyter session the archive is written to disk. Alternatively set `RUN_DIR` to persistent storage before the run. This cell does not require Google Drive or a repository.

In [ ]:
DOWNLOAD_RESULTS, INCLUDE_CHECKPOINTS = False, False
if DOWNLOAD_RESULTS:
    import zipfile
    archive = RUN_DIR.parent / f"{ALGORITHM}_results.zip"
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for path in RUN_DIR.rglob("*"):
            if path.is_file() and (INCLUDE_CHECKPOINTS or path.suffix in (".json", ".jsonl", ".png")):
                bundle.write(path, path.relative_to(RUN_DIR))
    try:
        from google.colab import files
        files.download(str(archive))
    except ImportError:
        print("Saved archive:", archive)

## References

Read the [OPSD method paper](https://arxiv.org/abs/2601.18734) for motivation and the [Qwen/Qwen3.5-0.8B model card](https://huggingface.co/Qwen/Qwen3.5-0.8B) for the selected architecture. The equations above describe this embedded implementation and its stated adaptations. Dataset documentation is available in the [GSM8K card](https://huggingface.co/datasets/openai/gsm8k) and [SVAMP card](https://huggingface.co/datasets/ChilleD/SVAMP). The notebook is sufficient by itself; these links provide background, not executable dependencies.

For a subsequent controlled experiment, change one training variable, use validation to assess it, keep a fixed final test protocol, and report the actual data counts, model/data revisions, hardware, seed, decoding budget, and any deviations.

Dataset protocols: [MBPP](https://huggingface.co/datasets/google-research-datasets/mbpp), [HumanEval](https://huggingface.co/datasets/openai/openai_humaneval), [Hermes function calling](https://huggingface.co/datasets/NousResearch/hermes-function-calling-v1), [xLAM](https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k), and [BFCL data](https://huggingface.co/datasets/gorilla-llm/Berkeley-Function-Calling-Leaderboard). For official benchmark reproduction use each publisher's documented protocol; this notebook labels its coding/tool adaptations explicitly.